# Submit 13 to RSNA Knee Abnormality Detection: our five-family stack (submission 11b) rank-blended with Raptor-384

Three stages, each timed:

1. **Raptor-384** (nartaa's published 0.949 accuracy pipeline, run unchanged in its own process with a 2 h timeout, so its GPU memory is released before stage 2). It writes `raptor_raw.csv` (per-study probabilities; a study it failed on stays 0.5 in every column) and `raptor_sub.csv` (its own rank table), never `submission.csv`.
2. **Our stack**: submission 11b's members, caches, scoring and family blend, byte-identical except three edits: the clock is a total-notebook deadline of 8.5 h measured from the first cell (so the Raptor stage's time is counted), the time trimmer is the family-safe one from submission 12, and `PRIORITY` puts the slot-ConvNeXt sets and the EXP-045 CoAtNet at the bottom (their board value is at most 0). Submission 11b's public-stage cells were off there and are not carried.
3. **Blend**: per finding, each side is percentile-ranked over its own non-failed studies, `final = 0.5 * rank(ours) + 0.5 * rank(Raptor)`, flat over the twelve findings and fixed before any run. A study one side failed takes the other side's rank. If the Raptor stage fails, the submission is our stack alone, exactly as submission 11b writes it, and the log carries a `BLEND_FALLBACK raptor` line. `submission.csv` is checked (row count, twelve label columns, no NaN, ids equal to `test.csv`) before the kernel ends.

No weight in this notebook was fitted on leaderboard feedback.

## Credits and required acknowledgement

Built on dreaddevelopment's [Raptor CoAtNet/MIL inference recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). These are our retrained checkpoints, rather than those baseline weights. Weak-label sources include [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), combined with our report-derived labels. Source reports and study-level label tables are not included here. The CoAtNet backbone comes from timm. Competition data is provided under the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).

## Stage 1: Raptor-384, nartaa's scored pipeline

The next cell writes [`nartaa/rsna-knee-0949-anatomical-mirror`](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror)'s two code cells to a script, in order: its checkpoint check (the attached `raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt` must have SHA-256 `7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef`) and its scored source (SHA-256 `bcd864cbf349882e876959b49cb348cbb89b4c62fc204d4e4261049c5e3a79c5` as published). The source is unchanged except for two marked `[S13]` edits in `main()`: it also writes the raw probabilities to `raptor_raw.csv`, and it writes its own table to `raptor_sub.csv` instead of `submission.csv`. Weights: [`nartaa/rsna-knee-publication-swa-weights-20261007`](https://www.kaggle.com/datasets/nartaa/rsna-knee-publication-swa-weights-20261007), version 1, used under its stated terms (research and educational use including this competition; non-commercial adaptation; the OAI notice below kept).

In [ ]:
# Submission 13: the notebook clock and the fixed settings of stages 1 and 3.
import time
from pathlib import Path

S13 = {
    "t0": time.time(),            # stage 2's deadline (BUDGET_H) is measured from here
    "raptor_script": Path("/tmp/s13_raptor.py"),
    "raptor_timeout_s": 2 * 3600,
    # Raptor's two outputs, written to its hard-coded /kaggle/working; neither name is one of ours.
    "raptor_files": ("raptor_raw.csv", "raptor_sub.csv"),
    "w_ours": 0.5,                # flat over the twelve findings, fixed before any run
    "raptor_ok": None,
    "raptor_error": None,
    "times": {},
}
print("S13 kernel start", time.strftime("%Y-%m-%d %H:%M:%S"),
      f"| blend {S13['w_ours']:.2f} ours + {1 - S13['w_ours']:.2f} Raptor"
      f" | Raptor timeout {S13['raptor_timeout_s'] / 3600:.1f} h")

In [ ]:
%%writefile /tmp/s13_raptor.py
# Authorized scoring run: verify the selected checkpoint before inference.
RUN_INFERENCE = True
if not RUN_INFERENCE:
    raise SystemExit("Static notebook only. Attach the exact checkpoint and competition, enable a T4 GPU, and set RUN_INFERENCE = True to run inference.")

import hashlib
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("This scored pipeline requires CUDA; CPU inference is not enabled.")
_weight_name = 'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt'
_candidates = [
    Path('/kaggle/input/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/rsna-knee-oai-original384-s42-pair') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-oai-original384-s42-pair') / _weight_name,
]
_weights = next((p for p in _candidates if p.is_file()), None)
if _weights is None:
    raise FileNotFoundError("Attach the exact selected weights; no alternative model is substituted.")
with _weights.open('rb') as _f:
    assert hashlib.file_digest(_f, 'sha256').hexdigest() == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', "Checkpoint hash differs"

# %% [code]
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import threading, queue
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
INPUT_RES = 320
INPUT_MODE = "crop"
IMG = 384   # native 384 corpus: the model saw 384 px crops directly, no 336->384 upsample
CROP_MM = 140.0
# 96 slices per study at native 384 px. Must match the corpus the weights were trained on
# (cache/build_corpus96.py).
SLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18)]
MAXS = sum(s[2] for s in SLOTS)                     # 96
K_EVAL = 94   # every window position of the 96-slice test volume (full coverage)
HDR_THREADS = 16   # per prep process: header reads are fuse-latency-bound, not CPU-bound
PX_THREADS = 8     # per prep process: pixel reads of the ~96 picked slices
QUEUE_MAX = 48     # studies in flight between the prep pool and the GPU consumers (RAM bound: ~14 MB each)
NORM = "imagenet"
FASTREAD_VARIANT = 'study_lut'   # kernel v3: research/fastread/readers.py, byte-identical to build_study
FR_HDR, FR_PX = 32, 16
_FASTREAD_B64 = "..."
def _load_fastread():
    import base64 as _b, importlib.util as _iu
    p = '/kaggle/working/_fastread.py' if os.path.isdir('/kaggle/working') else os.path.join(os.path.dirname(os.path.abspath(__file__)), '_fastread.py')
    if not os.path.exists(p):
        open(p, 'wb').write(_b.b64decode(_FASTREAD_B64))
    spec = _iu.spec_from_file_location('_fastread', p); m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
    return m
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [   # our 5-fold CoAtNet (public recipe reproduced on our folds/labels), equal-weight rank-mean
    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "w": 1.0},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3, img_size=INPUT_RES)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        # Bound backbone memory; the attention head still sees every window together.
        flat = x.flatten(0, 1)
        f = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    import hashlib
    with open(pt_path, "rb") as stream:
        observed_sha = hashlib.file_digest(stream, "sha256").hexdigest()
    assert observed_sha == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', "Bound checkpoint bytes differ"
    print(f"[weights] sha256 {observed_sha}", flush=True)
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    assert int(ck.get("res", res_default)) == 384, "Expected original 384 checkpoint"
    ck_res = INPUT_RES
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


SLOT_SIZES = [26, 22, 18, 12, 18]
SAGITTAL_SLOTS = (0, 1)

def _slot_index(D):
    b = np.cumsum([0] + SLOT_SIZES)
    if b[-1] != D: raise RuntimeError(f'slot sizes sum to {b[-1]}, volume has {D} slices')
    s_ = np.searchsorted(b, np.arange(D), side='right') - 1
    rel = (np.arange(D) - b[s_]) / np.maximum(1, np.array(SLOT_SIZES)[s_] - 1)
    return s_, rel

def _mirror_tri(tri, cs, D):
    """final-g1-bundle MIRROR_TTA: true L/R mirror of (k,3,H,W) uint8 eval triplets (research/rnd4/mirror/mirror_f0.py). The slot of each
    window's CENTRE slice decides the axis: sagittal (slots 0,1) -> reverse the 3 slice-channels; coronal/axial (2,3,4) -> flip the width.
    Done on uint8 BEFORE the per-channel LUT, i.e. 'mirror the knee, then normalise'."""
    s_, _ = _slot_index(D)
    sag = torch.from_numpy(np.isin(s_[np.asarray(cs, dtype=np.int64)], SAGITTAL_SLOTS)).to(tri.device).view(-1, 1, 1, 1)
    return torch.where(sag, tri.flip(1), tri.flip(-1))

def infer_pair_probs(model, plain, mirrored, device):
    assert plain.shape==mirrored.shape==(94,3,320,320)
    assert all(not m.training for m in model.modules()), "TTA must not update BN or enable dropout"
    p = infer_probs(model, plain, device)
    q = infer_probs(model, mirrored, device)
    return 0.5 * (p + q)

def eval_windows(vol, mask, k, res, norm=NORM, mirror=False):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0)
        if mirror:
            tri = _mirror_tri(torch.from_numpy(tri)[None], [c], D)[0].numpy()
        tri = tri.astype(np.float32) / 255.0
        assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
        lo = (IMG - res) // 2
        tri = tri[:, lo:lo + res, lo:lo + res]
        t = torch.from_numpy(np.ascontiguousarray(tri))
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


_WLUT = {}
def gpu_windows(vol, mask, k, res, dev, norm=NORM, mirror=False):
    """Same windows as eval_windows, built on the GPU from the uint8 volume. Values come from a per-channel
    256-entry table computed on the CPU with eval_windows' own float32 ops, so they are bit-identical."""
    D = vol.shape[0]
    cs = np.array([max(1, min(c, D - 2)) for c in _eval_centers(mask, D, k)])
    tri = torch.from_numpy(np.ascontiguousarray(vol[np.stack([cs - 1, cs, cs + 1], 1)])).to(dev)   # (k,3,IMG,IMG) uint8
    if mirror:
        tri = _mirror_tri(tri, cs, D)
    assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
    lo = (IMG - res) // 2
    tri = tri[:, :, lo:lo + res, lo:lo + res]
    key = (str(dev), norm)
    if key not in _WLUT:
        v = torch.from_numpy((np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0))
        t = torch.stack([v, v, v], 0).view(3, 256, 1)
        if norm == 'imagenet':
            t = (t - _MEAN.view(3, 1, 1)) / _STD.view(3, 1, 1)
        _WLUT[key] = t.view(3, 256).contiguous().to(dev)
    L = _WLUT[key]
    x = torch.stack([L[c][tri[:, c].long()] for c in range(3)], 1)                   # exact float32 values
    if x.shape[-1] != res:
        if norm == 'imagenet':   # resize must happen BEFORE normalisation, as in eval_windows
            L0 = _WLUT.setdefault((str(dev), 'none'), torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0).to(dev))
            x = L0[tri.long()]
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
            x = (x - _MEAN.to(dev)) / _STD.to(dev)
        else:
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    assert str(device).startswith("cuda"), "Qualified inference requires CUDA/fp16"
    x = xwins.unsqueeze(0).to(device)
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(x).float()
        assert torch.isfinite(logits).all(), "Nonfinite fp16 logits"
        o = torch.sigmoid(logits)
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def _hdr(f):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            iop = getattr(h, 'ImageOrientationPatient', None)
            ipp = getattr(h, 'ImagePositionPatient', None)
            if iop is not None and ipp is not None and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(getattr(h, 'InstanceNumber', 0) or 0)
            ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
            return (pos, f, ps, True)
        except Exception:
            return (0.0, f, 0.5, False)

    def order_and_meta(sdir):
        # Header reads are latency-bound on Kaggle's mounted competition data (~240k opens on the hidden
        # set); a thread pool hides that latency. Records and sort are unchanged -> identical ordering.
        fs = glob.glob(sdir + "/*.dcm")
        with ThreadPoolExecutor(max_workers=HDR_THREADS) as ex:
            out = list(ex.map(_hdr, fs))
        recs = [(pos, f, ps) for pos, f, ps, _ in out]; ps_list = [ps for _, _, ps, ok in out if ok]
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (build_corpus80.py, SPAN).
        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        sel = [files[min(p, n - 1)] for p in picks]
        def _px(fp_ps):
            fp, ps = fp_ps
            try: return read_px(fp), ps
            except Exception: return None, med_ps
        with ThreadPoolExecutor(max_workers=PX_THREADS) as ex:
            got = list(ex.map(_px, sel))                    # order preserved
        arrs = [a for a, _ in got]; pss = [ps for _, ps in got]
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask



# ---- parallel preprocessing helpers (module level so they pickle under fork) ----
_SER = None; _TSDIR = None; _READER = None
def _pre_init():
    global _READER
    _READER = _load_fastread().make_builder(FASTREAD_VARIANT, IMG, SLOTS, hdr_threads=FR_HDR, px_threads=FR_PX)
def _pre_one(sid):
    try:
        vol, mask = _READER(sid, _SER, _TSDIR)
    except Exception as e:
        print(f"  [pre] {sid[:16]} FAILED ({type(e).__name__}: {e})", flush=True)
        vol = np.zeros((MAXS, IMG, IMG), np.uint8); mask = np.zeros(MAXS, np.uint8)
    return sid, vol, mask

# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        # skip the competition data wherever Kaggle mounts it (/kaggle/input/competitions/<slug> OR /kaggle/input/<slug>):
        # recursively globbing its DICOM tree hung a commit run for 20+ min (2026-09-26)
        if "competition" in d.lower() or "abnormality-detection" in d or any(
                os.path.exists(os.path.join(d, x)) for x in ("test_series", "train_images", "train.csv", "test.csv")):
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]
    pos = {sid: i for i, sid in enumerate(test_ids)}

    # ---- 2. GPU consumers: one thread per GPU, every arm loaded on every GPU (per-study batch of 1,
    #         so the math is identical to the sequential kernel; only the wall-clock changes) ----
    xla = None
    try:
        import torch_xla; import torch_xla.core.xla_model as xm
        dev = torch_xla.device(); xla = xm
        print("device: XLA/TPU", dev, flush=True)
    except Exception as e:
        print("XLA unavailable:", type(e).__name__, str(e)[:80], "| device", dev, flush=True)
    devs = [f"cuda:{g}" for g in range(ngpu)] if (xla is None and ngpu > 0) else [dev]
    models = {}
    for d in devs:
        models[d] = []
        for a, arm in enumerate(ARMS):
            model, res = load_model(find_weight_file(arm["file"]), arm["arch"], arm["res"], d)
            models[d].append((model, res))
        print(f"[gpu {d}] loaded {A} arm(s) | {time.time()-t0:.0f}s", flush=True)
    q = queue.Queue(maxsize=QUEUE_MAX); done_n = [0]; lock = threading.Lock()

    def consumer(d):
        while True:
            item = q.get()
            if item is None: q.task_done(); return
            sid, vol, mask = item; i = pos[sid]
            try:
                for a, (model, res) in enumerate(models[d]):
                    xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM) if xla is not None else gpu_windows(vol, mask, K_EVAL, res, d)
                    assert xla is None, "Anatomical L/R TTA requires CUDA/fp16"
                    xm = gpu_windows(vol, mask, K_EVAL, res, d, norm=NORM, mirror=True)
                    arm_probs[a][i] = infer_pair_probs(model, xw, xm, d)
                    del xm
            except Exception as e:
                print(f"  [gpu {d}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            with lock:
                done_n[0] += 1
                if done_n[0] % 200 == 0 or done_n[0] == N:
                    print(f"  [gpu] {done_n[0]}/{N} | {time.time()-t0:.0f}s", flush=True)
            q.task_done()

    threads = [threading.Thread(target=consumer, args=(d,), daemon=True) for d in devs]
    for th in threads: th.start()

    # ---- 3. prep every study ONCE in a process pool and stream it to the GPUs as it lands ----
    import multiprocessing as mp
    global _SER, _TSDIR, _READER
    _SER, _TSDIR = SER, tsdir
    nproc = max(4, min(96, (os.cpu_count() or 8) - 8))
    t_pre = time.time()
    with mp.get_context("fork").Pool(nproc, initializer=_pre_init) as pool:
        for i, (sid, vol, mask) in enumerate(pool.imap_unordered(_pre_one, test_ids, chunksize=2)):
            q.put((sid, vol, mask))
            if (i + 1) % 200 == 0 or i + 1 == N:
                print(f"  [pre] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
    print(f"[pre] done: {N} studies in {time.time()-t_pre:.0f}s (procs {nproc} x fastread {FASTREAD_VARIANT} {FR_HDR}/{FR_PX} threads)", flush=True)
    for _ in devs: q.put(None)
    q.join()
    for th in threads: th.join()
    print(f"[gpu] done: {done_n[0]}/{N} studies on {len(devs)} device(s) | {time.time()-t0:.0f}s", flush=True)
    for d in devs:
        for m, _ in models[d]: del m
    models.clear(); gc.collect()
    # [S13] raw per-study probabilities for the notebook's blend; a study the GPU consumer failed on stays 0.5
    _s13_raw = pd.DataFrame(arm_probs[0].astype(np.float32), columns=LAB)
    _s13_raw.insert(0, "StudyInstanceUID", test_ids)
    _s13_raw.to_csv("/kaggle/working/raptor_raw.csv", index=False)

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/raptor_sub.csv"   # [S13] was submission.csv; the notebook's stage 3 writes that
    sub.to_csv(out, index=False)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# Stage 1: run the Raptor script in its own process group, stream its log here, stop it at the timeout.
# Any failure (non-zero exit, timeout, a missing output) marks the stage failed; stage 3 then writes
# our stack alone and prints BLEND_FALLBACK raptor.
import os
import shutil
import signal
import subprocess
import sys
import threading
import time
from pathlib import Path

_s13_override = globals().get("CFG_OVERRIDE") or {}
S13["work"] = Path(_s13_override.get("work_dir", "/kaggle/working"))
S13["work"].mkdir(parents=True, exist_ok=True)
for _f in S13["raptor_files"]:
    (S13["work"] / _f).unlink(missing_ok=True)      # nothing stale can pass as this run's output

_s13_t = time.time()
try:
    if _s13_override:
        # Mac dry run: the Raptor pipeline needs CUDA, so a stub table stands in for its outputs,
        # or, with no stub given, the stage fails and the fallback path runs.
        _stub = _s13_override.get("raptor_stub")
        if not _stub:
            raise RuntimeError("dry run without a Raptor stub")
        for _f in S13["raptor_files"]:
            shutil.copy2(Path(_stub) / _f, S13["work"] / _f)
        _rc = 0
    else:
        _proc = subprocess.Popen([sys.executable, "-u", str(S13["raptor_script"])],
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                 bufsize=1, start_new_session=True)

        def _s13_pump(stream):
            for line in stream:
                print("[raptor] " + line, end="", flush=True)

        _pump = threading.Thread(target=_s13_pump, args=(_proc.stdout,), daemon=True)
        _pump.start()
        try:
            _rc = _proc.wait(timeout=S13["raptor_timeout_s"])
        except subprocess.TimeoutExpired:
            os.killpg(_proc.pid, signal.SIGKILL)
            _proc.wait()
            raise RuntimeError(f"timed out after {S13['raptor_timeout_s'] / 3600:.1f} h")
        finally:
            try:
                os.killpg(_proc.pid, signal.SIGKILL)  # prep workers left behind, if any
            except (ProcessLookupError, PermissionError):
                pass
            _pump.join(timeout=30)
    if _rc != 0:
        raise RuntimeError(f"exit code {_rc}")
    _missing = [_f for _f in S13["raptor_files"] if not (S13["work"] / _f).is_file()]
    if _missing:
        raise RuntimeError(f"missing outputs {_missing}")
    S13["raptor_ok"] = True
except Exception as exc:
    S13["raptor_ok"] = False
    S13["raptor_error"] = f"{type(exc).__name__}: {exc}"
    print("=" * 78)
    print(f"S13 RAPTOR STAGE FAILED: {S13['raptor_error'][:2000]}")
    print("=" * 78)
S13["times"]["raptor"] = time.time() - _s13_t
_s13_el = time.time() - S13["t0"]
print(f"RAPTOR_STAGE ok={S13['raptor_ok']} | {S13['times']['raptor'] / 60:.1f} min | "
      f"{_s13_el / 3600:.2f} h into the notebook", flush=True)
if shutil.which("nvidia-smi") and not _s13_override:
    subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader"], check=False)

## Stage 2: our stack, submission 11b

The cells from here to *Stage 3* are submission 11b's (public LB 0.951), unchanged except for three edits in the next two cells, each marked `[S13]`:

- `NB_T0` is the notebook start from stage 1's clock, and `BUDGET_H = 8.5` is a total-notebook deadline, so the pre-build plan and the probe-time trimmer both see the time the Raptor stage took.
- `PRIORITY` keeps the series-bag sets, exp063 and the exp053 CoAtNet twin longest; the slot-ConvNeXt sets (036a, 036b, 044a, then 037b, 041) and exp045 are at the bottom. `*sub10*` is listed beside `*exp056*`, because on Kaggle the flat sub10 dataset names exp056's set `rsna-knee-weights-sub10`, which matched no pattern in 11b and made it the trimmer's first drop. Its family is unchanged (it sits in `convnext_bag`, as in 11b's scored run).
- `trim_members` is submission 12's family-safe version: a set that is its family's last is passed over while another family still has two.

Submission 11b's members, caches, TTA plan (`[0, 1]`, cut to one pass by the probe on the hidden test), scoring and family blend are otherwise as scored. Its public-stage cells (off in 11b, `PUBLIC_STAGE = False`) are not carried, and its write-out cells are replaced by stage 3, which writes exactly 11b's table when Raptor is unavailable.

In [ ]:
import os
import sys
import time
import fnmatch
import traceback
from pathlib import Path

NB_T0 = S13["t0"] if "S13" in globals() else time.time()   # [S13] the notebook start
print("kernel start", time.strftime("%Y-%m-%d %H:%M:%S"))

# ---- Members ---------------------------------------------------------------------------------
# Our weight datasets (S6 hygiene e: nothing else under /kaggle/input is searched for
# checkpoints). Each holds one directory per weight set, with best_fold<k>.pt / swa_fold<k>.pt
# inside; a directory ending in _full holds one full-data checkpoint. A set present in two
# datasets is scored once.
WEIGHT_OWNER = "darcyjosephdghg"
WEIGHT_SLUGS = ["rsna-knee-weights-sub11", "rsna-knee-weights-sub10", "rsna-knee-weights-sub8", "rsna-knee-weights-sub7",
                "rsna-knee-weights-sub5"]

# Weight-set directory name (fnmatch pattern) -> family; the first matching pattern wins. A set
# that matches nothing is put in the family named by its backbone's leading word ("convnext",
# "coatnet", "resnet", with "_bag" appended for a cache-v5 series-bag model), with a warning.
FAMILY = {
    "*exp036*": "convnext",
    "*exp037*": "convnext",
    "*exp039*": "convnext",
    "*exp041*": "convnext",
    "*exp043*": "convnext",
    "*exp044*": "convnext",
    "*exp045*": "coatnet",
    "*exp046*": "resnet",
    "*exp047*": "convnext_bag",
    "*exp048*": "convnext_bag",
    "*exp049*": "convnext_bag",
    "*exp050*": "convnext_bag",
    "*exp051*": "convnext_bag",
    "*exp053*": "coatnet",
    "*exp054*": "coatnet_bag",
    "*exp055*": "convnext_bag",
    "*exp056*": "convnext_bag56",
    "*exp063*": "convnext_bag63",
}

# Weight sets (fnmatch on the set directory name) left out of this submission even if attached.
# S11b (ledger EXP-063 verdict): S11a + exp063 as a fifth family; exp064 is attached (sub11) but left out.
EXCLUDE_SETS = ["*exp047*", "*exp049*", "*exp054*", "*exp064*"]

# Weight of each family in the across-family rank mean; a family not listed gets 1, so the
# default is equal weight per family. Set before any score is seen, never tuned on the board.
FAMILY_WEIGHT = {}

# What the time trimmer keeps longest, highest priority first: fnmatch patterns on the set name
# or on the family. Sets that match no pattern are dropped first, then sets from the end of
# this list upward; the last set is never dropped.
# Pre-registered 2026-10-03 13:40 (ledger SUBMISSION 7): kept longest = strongest offline evidence.
# [S13] ledger SUBMISSION 13: the slot-ConvNeXt sets and exp045 go first (board value <= 0: S8a 0.947 vs
# S9a 0.948); *sub10* is exp056's set as Kaggle names it (flat dataset), listed beside *exp056*.
PRIORITY = ["*exp055*", "*exp051*", "*exp056*", "*sub10*", "*exp063*", "*exp053*", "*exp041*", "*exp037b*", "*exp045*",
            "*exp044*", "*exp036a*", "*exp036b*"]

# ---- Time --------------------------------------------------------------------------------------
# Wall clock from the kernel start to the end of scoring; Kaggle stops the kernel at 9 h.
# [S13] A total-notebook deadline from the first cell, so the Raptor stage's time is counted.
BUDGET_H = 8.5
TTA_OFFSETS = [0, 1]
# Studies timed per group after one untimed warm-up study.
PROBE_STUDIES = 8
# Pre-build cost hints, seconds per study on 2 x T4: per member and TTA pass (submission 5
# measured 0.285 s for a ConvNeXt-Tiny 320 checkpoint), and per cache built alone on 4 CPUs
# (v4 and v5 builder logs). Used only to drop sets before their caches are built; the run
# itself is sized from measured times.
PASS_HINT_S = {"convnext": 0.285, "coatnet": 0.71, "resnet": 0.086}
BUILD_HINT_S = {"v3": 1.1, "v4": 1.33, "v5": 1.46}
# Backbones fed this side or more run one TTA view per forward call (memory, not results).
ONE_VIEW_MIN_SIDE = 384

# ---- Optional public stage: submission 6's d4 stack, unchanged ----------------------------------
PUBLIC_STAGE = False
OUR_WEIGHT = 0.45
ALLOW_FALLBACK = True
PUBLIC_TTA2_MAX_H = 2.5
# Hours the public stage is assumed to take when the caches are planned before it runs.
PUBLIC_EST_H = 2.5

# knee.infer.dryrun --notebook defines CFG_OVERRIDE: the Mac dry run. Its comp_dir is either a
# fake competition mount (then the caches are built from its DICOMs) or not (then pre-built
# caches are read from work_dir/s7_cache/<name>, or work_dir/cache for v3-320).
_override = globals().get("CFG_OVERRIDE") or {}
DRY_RUN = bool(_override)
if DRY_RUN:
    PUBLIC_STAGE = False
    # The dry run can shrink the budget and plan for more studies, to exercise the trimmer.
    BUDGET_H = float(_override.get("budget_h", BUDGET_H))

assert 0.0 <= OUR_WEIGHT <= 1.0, OUR_WEIGHT
assert all(w >= 0 for w in FAMILY_WEIGHT.values()), FAMILY_WEIGHT

# Everything that must outlive the public stage lives in this one dict: the public cells run in
# this namespace and rebind common names (ROOT, TARGETS, CFG, log, time, ...).
_work = Path(_override.get("work_dir", "/kaggle/working"))
S7 = {
    "stage": "both" if PUBLIC_STAGE else "ours",
    "our_weight": OUR_WEIGHT,
    "allow_fallback": ALLOW_FALLBACK and PUBLIC_STAGE,
    "public_tta2_max_h": PUBLIC_TTA2_MAX_H,
    "dry_run": DRY_RUN,
    "dry_run_studies": int(_override.get("studies", 0) or 0),
    "nb_t0": NB_T0,
    "work_dir": _work,
    "cache_root": (_work / "s7_cache") if DRY_RUN else Path("/tmp/s7_cache"),
    "weights_dir": Path(_override["weights"]) if DRY_RUN else None,
    "comp_dir": Path(_override.get("comp_dir", "/nonexistent")) if DRY_RUN else None,
    "comp_root": None,
    "tta_offsets": TTA_OFFSETS[:max(1, int(_override.get("tta", 2)))] if DRY_RUN else list(TTA_OFFSETS),
    "public_active": PUBLIC_STAGE,
    "public_ok": None,
    "public_error": None,
    "ours_active": True,
    "ours_ok": None,
    "ours_error": None,
    "times": {},
    "n_studies": {},
    "dropped": [],
}
S7["work_dir"].mkdir(parents=True, exist_ok=True)
# The public stage's cells are submission 6's, verbatim; they read S6["public_active"] and call
# _s6_public_failed. S6 is the same dict as S7.
S6 = S7


def _s7_stage_failed(half, exc):
    """Records a failed half and either re-raises or falls back to the other half.

    Args:
        half: "public" or "ours".
        exc: The exception the half raised.

    Raises:
        BaseException: `exc` again, when fallback is off.
    """
    import traceback as _tb

    S7[f"{half}_active"] = False
    S7[f"{half}_ok"] = False
    S7[f"{half}_error"] = f"{type(exc).__name__}: {exc}"
    print("=" * 78, flush=True)
    print(f"S7 {half.upper()} STAGE FAILED: {S7[f'{half}_error'][:2000]}", flush=True)
    _tb.print_exception(type(exc), exc, exc.__traceback__, limit=6)
    print("=" * 78, flush=True)
    if not S7["allow_fallback"]:
        raise exc
    other = "ours" if half == "public" else "public"
    print(f"S7 FALLBACK: the {half} stage is dropped; the submission is the {other} stage alone",
          flush=True)


def _s6_public_failed(exc):
    _s7_stage_failed("public", exc)


def _s7_ours_failed(exc):
    _s7_stage_failed("ours", exc)


def _s7_comp_root():
    """Finds the competition mount (the directory holding sample_submission.csv and test.csv).

    Returns:
        The competition directory.

    Raises:
        FileNotFoundError: No mounted directory holds both files.
    """
    if S7["comp_root"] is not None:
        return Path(S7["comp_root"])
    if S7["dry_run"]:
        candidates = [S7["comp_dir"]]
    else:
        candidates = [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                      Path("/kaggle/input/rsna-knee-abnormality-detection")]
        candidates += [p.parent for p in sorted(Path("/kaggle/input").glob("*/sample_submission.csv"))]
        candidates += [p.parent for p in sorted(Path("/kaggle/input").glob("*/*/sample_submission.csv"))]
    for c in candidates:
        if (c / "sample_submission.csv").is_file() and (c / "test.csv").is_file():
            S7["comp_root"] = c
            return c
    raise FileNotFoundError("no mounted directory holds sample_submission.csv and test.csv")


print(f"PUBLIC_STAGE={PUBLIC_STAGE} | budget {BUDGET_H} h | TTA {S7['tta_offsets']} | "
      f"dry run={DRY_RUN} | work dir {S7['work_dir']} | caches under {S7['cache_root']}")
print("python", sys.version.split()[0], "| cpus:", os.cpu_count())

## Plan: members, input contracts, families

The functions in the next cell are `knee/predict.py`'s, copied verbatim (a kernel cannot import the repo). They read each checkpoint's stored training config straight from the zip without torch, so no CUDA context or torch thread pool exists yet when the cache workers fork.

A member's **input contract** is: the cache it reads (version from `CACHE_KEY`, side `IMG`, slices `CACHE_SLICES`; v5 when `CACHE_FORMAT` is `series`), the side its backbone is fed (`MODEL_IMG`, else the cache side), `SLICE_STRIDE`, and the slot subset (`SLOTS`) or bag size and layout (`N_SERIES_MAX`, `SERIES_LAYOUT`). Members with equal contracts form one group and share one read per study.

In [ ]:
import fnmatch
import hashlib
import io
import pickle
import re
import zipfile
from dataclasses import dataclass

import numpy as np
import pandas as pd

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

# --- verbatim from src/knee/predict.py: MODEL_DEFAULTS, INFER_OVERRIDES, CACHE_RULES, PIXEL_MEAN, CacheContract, InputContract, _StubTensor, _ConfigUnpickler, read_checkpoint_cfg, member_cfg, cache_contract, input_contract, Member, family_of, discover_members, group_members, projected_seconds, _priority, trim_members
MODEL_DEFAULTS = {
    "N_SLOTS": 6, "ATTN_DIM": 128, "DROPOUT": 0.2, "HEAD": "concat", "HEAD_HIDDEN": 256,
    "SLOT_EMB": False, "POS_EMB": False, "SLICE_STRIDE": 2, "IMG": 320, "MODEL_IMG": None,
    "CACHE_SLICES": 24, "CACHE_KEY": "rsna-knee-cache-v3", "SLOTS": None, "VIT_IMG": 224,
    "CACHE_FORMAT": "slots", "N_SERIES_MAX": 8, "SERIES_LAYOUT": "typed", "META_EMB": True,
    "DEPTH_BINS": 16,
}

INFER_OVERRIDES = {"PRETRAINED": False, "GRAD_CHECKPOINT": False}

CACHE_RULES = {
    "v3": {"layout": "slots", "band": (0.10, 0.90), "fs_negation": False,
           "instance_fallback": False},
    "v4": {"layout": "slots", "band": (0.10, 0.90), "fs_negation": True,
           "instance_fallback": True},
    "v5": {"layout": "series", "band": (0.02, 0.98), "fs_negation": True,
           "instance_fallback": True},
}

PIXEL_MEAN, PIXEL_STD = 0.449, 0.226

@dataclass(frozen=True)
class CacheContract:
    """Which inline test cache a member reads.

    Attributes:
        rules: Cache version whose build rules apply: 'v3', 'v4' or 'v5'.
        img: Cache side in pixels.
        n_slices: Slices stored per slot or series.
    """

    rules: str
    img: int
    n_slices: int

    @property
    def layout(self) -> str:
        """The npz layout of this cache.

        Returns:
            'slots' or 'series'.
        """
        return CACHE_RULES[self.rules]["layout"]

    @property
    def name(self) -> str:
        """A short name, also used as the cache directory name.

        Returns:
            For example 'v3-320', or 'v3-320-s32' when the slice count is not the version's
            usual one.
        """
        usual = 32 if self.layout == "series" else 24
        return f"{self.rules}-{self.img}" + ("" if self.n_slices == usual else
                                             f"-s{self.n_slices}")

@dataclass(frozen=True)
class InputContract:
    """Everything that decides how a cached study becomes one member's input.

    Two checkpoints with equal contracts read identical tensors, so they share one load per
    study.

    Attributes:
        cache: The test cache read.
        side: Image side the backbone is fed (MODEL_IMG, else the cache side).
        stride: Every stride-th stored slice is a sample position.
        slots: Slot subset of a slot cache, or None for every slot.
        n_series: Bag size of a series cache, or None for a slot cache.
        typed: Series cache: typed bag layout (one reserved position per series type).
    """

    cache: CacheContract
    side: int
    stride: int
    slots: tuple[int, ...] | None = None
    n_series: int | None = None
    typed: bool = True

    @property
    def name(self) -> str:
        """A short readable name for logs.

        Returns:
            For example 'v4-384@320/s2' or 'v5-288@288/s2/bag8typed'.
        """
        out = f"{self.cache.name}@{self.side}/s{self.stride}"
        if self.slots is not None:
            out += "/slots" + "".join(str(k) for k in self.slots)
        if self.n_series is not None:
            out += f"/bag{self.n_series}{'typed' if self.typed else 'packed'}"
        return out

class _StubTensor:
    """Stands in for every torch object while a checkpoint's config is unpickled."""

    def __init__(self, *args, **kwargs):
        pass

    def __call__(self, *args, **kwargs):
        """Returns None for any rebuild call.

        Args:
            *args: Ignored.
            **kwargs: Ignored.

        Returns:
            None.
        """
        return None

class _ConfigUnpickler(pickle.Unpickler):
    """Unpickles a torch checkpoint's dict without torch: tensors come back as None."""

    def find_class(self, module, name):
        """Resolves a pickled global, replacing anything from torch with a stub.

        Args:
            module: The global's module.
            name: The global's name.

        Returns:
            The class or function, or a stub for torch objects.
        """
        if module.split(".")[0] == "torch":
            return _StubTensor()
        return super().find_class(module, name)

    def persistent_load(self, pid):
        """Skips the tensor storages.

        Args:
            pid: The storage's persistent id.

        Returns:
            None.
        """
        return None

def read_checkpoint_cfg(path) -> dict:
    """Reads the training config stored in a checkpoint without loading its weights.

    Works on the zip format torch.save writes, without importing torch, so the kernel can plan
    its caches before any CUDA context or thread pool exists in the process that forks the
    cache workers.

    Args:
        path: A checkpoint written by `knee.train`.

    Returns:
        The stored training config, with the checkpoint's `ema` flag under `_EMA`.

    Raises:
        ValueError: The file holds no single data.pkl or carries no `cfg`.
    """
    try:
        with zipfile.ZipFile(path) as z:
            names = [n for n in z.namelist() if n.endswith("data.pkl")]
            if len(names) != 1:
                raise ValueError(f"{path}: expected one data.pkl in the checkpoint, got {names}")
            d = _ConfigUnpickler(io.BytesIO(z.read(names[0]))).load()
    except zipfile.BadZipFile:
        # A checkpoint in torch's legacy (non-zip) format: load it whole.
        import torch as _torch

        d = _torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(d, dict) or "cfg" not in d:
        raise ValueError(f"{path} carries no training config")
    cfg = dict(d["cfg"])
    cfg["_EMA"] = bool(d.get("ema", False))
    return cfg

def member_cfg(raw: dict) -> dict:
    """Completes a stored training config the way `model_from_checkpoint` does.

    Args:
        raw: The config as stored in the checkpoint.

    Returns:
        MODEL_DEFAULTS, overlaid with the stored non-None values and INFER_OVERRIDES.
    """
    cfg = dict(MODEL_DEFAULTS)
    cfg.update({k: v for k, v in raw.items() if v is not None})
    cfg.update(INFER_OVERRIDES)
    return cfg

def cache_contract(cfg: dict) -> CacheContract:
    """Names the test cache a member was trained on.

    The version is the 'v<digits>' in CACHE_KEY ('cache_v4', 'rsna-knee-cache-v3'); a series
    cache (CACHE_FORMAT 'series') must be v5 and a slot cache must not.

    Args:
        cfg: A completed member config (`member_cfg`).

    Returns:
        The cache contract.

    Raises:
        ValueError: The version cannot be read, is unknown, or disagrees with CACHE_FORMAT.
    """
    m = re.search(r"v(\d+)", str(cfg.get("CACHE_KEY", "")))
    if m is None:
        raise ValueError(f"no cache version in CACHE_KEY {cfg.get('CACHE_KEY')!r}")
    rules = f"v{m.group(1)}"
    if rules not in CACHE_RULES:
        raise ValueError(f"cache {rules} has no inline build rules")
    fmt = cfg.get("CACHE_FORMAT", "slots")
    if (fmt == "series") != (CACHE_RULES[rules]["layout"] == "series"):
        raise ValueError(f"CACHE_FORMAT {fmt!r} does not match cache {rules}")
    return CacheContract(rules, int(cfg["IMG"]), int(cfg["CACHE_SLICES"]))

def input_contract(cfg: dict) -> InputContract:
    """Reads a member's input contract from its completed config.

    Args:
        cfg: A completed member config (`member_cfg`).

    Returns:
        The input contract.
    """
    cache = cache_contract(cfg)
    side = cfg["VIT_IMG"] if "vit" in cfg["BACKBONE"] else (cfg.get("MODEL_IMG") or cfg["IMG"])
    if cache.layout == "series":
        return InputContract(cache, int(side), int(cfg["SLICE_STRIDE"]),
                             n_series=int(cfg["N_SERIES_MAX"]),
                             typed=cfg.get("SERIES_LAYOUT", "typed") == "typed")
    slots = tuple(int(k) for k in cfg["SLOTS"]) if cfg.get("SLOTS") is not None else None
    return InputContract(cache, int(side), int(cfg["SLICE_STRIDE"]), slots=slots)

@dataclass
class Member:
    """One checkpoint of the ensemble and what the kernel needs to know about it.

    Attributes:
        path: The checkpoint file.
        set_name: Name of the directory holding its fold set (the weight-set name).
        fold: Its fold index (0 for a full-data twin).
        cfg: The completed training config.
        contract: Its input contract.
        family: The family it is blended in.
    """

    path: Path
    set_name: str
    fold: int
    cfg: dict
    contract: InputContract
    family: str

def family_of(set_name: str, cfg: dict, families: dict) -> str:
    """Finds the family a weight set belongs to.

    Args:
        set_name: The weight-set directory name.
        cfg: The completed config of one of its checkpoints.
        families: fnmatch pattern on the set name -> family name; the first match wins.

    Returns:
        The family. When no pattern matches, the backbone's leading word ('convnext',
        'coatnet', 'resnet'), with '_bag' appended for a series-bag model, and a printed
        warning.
    """
    for pattern, family in families.items():
        if fnmatch.fnmatchcase(set_name, pattern):
            return family
    m = re.match(r"[a-z]+", str(cfg["BACKBONE"]).lower())
    fallback = (m.group(0) if m else str(cfg["BACKBONE"])) + (
        "_bag" if cfg.get("CACHE_FORMAT") == "series" else "")
    print(f"WARNING: weight set {set_name} matches no FAMILY pattern; family {fallback!r}")
    return fallback

def discover_members(paths, families: dict) -> list[Member]:
    """Describes every checkpoint: its config, input contract and family.

    Args:
        paths: Checkpoint paths, fold-major as `find_checkpoints` returns them.
        families: The FAMILY mapping (see `family_of`).

    Returns:
        One Member per readable path, in the order given. A checkpoint whose config cannot be
        read or names no servable input is skipped with a printed line, not fatal.
    """
    out, fam_cache = [], {}
    for path in paths:
        path = Path(path)
        try:
            cfg = member_cfg(read_checkpoint_cfg(path))
            contract = input_contract(cfg)
        except (ValueError, KeyError, OSError, pickle.UnpicklingError) as exc:
            print(f"SKIPPED {path.parent.name}/{path.name}: {type(exc).__name__}: {exc}")
            continue
        set_name = path.parent.name
        if set_name not in fam_cache:
            fam_cache[set_name] = family_of(set_name, cfg, families)
        fold = int(path.stem.partition("_fold")[2] or 0)
        out.append(Member(path, set_name, fold, cfg, contract, fam_cache[set_name]))
    return out

def group_members(members) -> dict[InputContract, list[Member]]:
    """Groups members by input contract, keeping the given order inside each group.

    Args:
        members: Members from `discover_members`.

    Returns:
        Contract -> its members, in order of first appearance.
    """
    groups: dict[InputContract, list[Member]] = {}
    for m in members:
        groups.setdefault(m.contract, []).append(m)
    return groups

def projected_seconds(members, cost, n_studies: int, n_offsets: int,
                      build_cost: dict | None = None) -> float:
    """Projects the seconds the scoring (and any unbuilt caches) still needs.

    Args:
        members: Members still in the ensemble.
        cost: Seconds per study per TTA pass, per member path.
        n_studies: Studies to score.
        n_offsets: TTA passes.
        build_cost: Cache name -> seconds to build it, for caches not yet built; a cache
            counts only while some member still reads it.

    Returns:
        Projected seconds.
    """
    total = sum(cost[m.path] for m in members) * n_offsets * n_studies
    if build_cost:
        needed = {m.contract.cache.name for m in members}
        total += sum(v for k, v in build_cost.items() if k in needed)
    return total

def _priority(member: Member, priority) -> int:
    """Position of a member's set in the PRIORITY list (lower is kept longer).

    Args:
        member: The member.
        priority: fnmatch patterns, matched against the set name and then the family.

    Returns:
        The index of the first matching pattern, or len(priority) when none matches.
    """
    for i, pattern in enumerate(priority):
        if (fnmatch.fnmatchcase(member.set_name, pattern)
                or fnmatch.fnmatchcase(member.family, pattern)):
            return i
    return len(priority)

def trim_members(members, cost, n_studies: int, seconds_left: float, n_offsets: int,
                 priority, build_cost: dict | None = None):
    """Sizes the ensemble to the clock: TTA passes first, then whole sets, then folds.

    TTA passes go first, as in submission 5 (a member is worth more than a second view).
    Then whole weight sets are dropped from the lowest priority up (sets matching no
    PRIORITY pattern are lowest; among equals the set found last goes first), never the
    last set. A set that is its family's last is passed over while another family still
    has two: a family that vanishes silently reweights the across-family mean, which is
    fixed before any score is seen. Once every family is down to one set, folds are
    dropped from the end of the lowest-priority set that still has two, never the last
    member; only when every set is a single member does the lowest-priority family go.

    Args:
        members: Members in discovery order.
        cost: Seconds per study per TTA pass, per member path.
        n_studies: Studies to score.
        seconds_left: Wall clock available.
        n_offsets: TTA passes wanted.
        priority: fnmatch patterns on set or family names, highest priority first.
        build_cost: As in `projected_seconds`.

    Returns:
        The kept members (in discovery order), the TTA passes to run, and a list of
        human-readable lines saying what was dropped.
    """
    kept, log = list(members), []

    def over(ms, offs):
        """Says whether an ensemble overruns the clock.

        Args:
            ms: Members.
            offs: TTA passes.

        Returns:
            True when the projection exceeds seconds_left.
        """
        return projected_seconds(ms, cost, n_studies, offs, build_cost) > seconds_left

    offsets = n_offsets
    while offsets > 1 and over(kept, offsets):
        offsets -= 1
        log.append(f"TTA {offsets + 1} -> {offsets} passes")
    order = list(dict.fromkeys(m.set_name for m in members))
    rank = {s: (_priority(next(m for m in members if m.set_name == s), priority), i)
            for i, s in enumerate(order)}
    family = {s: next(m.family for m in members if m.set_name == s) for s in order}

    def spare(names):
        """Lists the sets whose family would still be present without them.

        Args:
            names: Set names still kept, highest priority first.

        Returns:
            The names whose family has another set in `names`, in the same order.
        """
        n_sets = {}
        for s in names:
            n_sets[family[s]] = n_sets.get(family[s], 0) + 1
        return [s for s in names if n_sets[family[s]] > 1]

    sets = sorted(order, key=lambda s: rank[s])                     # highest priority first
    while len(sets) > 1 and over(kept, offsets):
        spare_sets = spare(sets)
        if not spare_sets and any(sum(m.set_name == s for m in kept) > 1 for s in sets):
            break          # every family is down to one set: shrink folds before deleting one
        gone = (spare_sets or sets)[-1]
        sets.remove(gone)
        n_gone = sum(m.set_name == gone for m in kept)
        kept = [m for m in kept if m.set_name != gone]
        log.append(f"dropped set {gone} ({n_gone} members, family {family[gone]})")
    while len(kept) > 1 and over(kept, offsets):
        by_set = {}
        for m in kept:
            by_set.setdefault(m.set_name, []).append(m)
        names = sorted(by_set, key=lambda s: rank[s])
        shrinkable = [s for s in names if len(by_set[s]) > 1 or s in spare(names)]
        victim = by_set[(shrinkable or names)[-1]][-1]
        kept.remove(victim)
        log.append(f"dropped {victim.set_name}/{victim.path.name}")
    return kept, offsets, log

In [ ]:
N_FOLDS = 5
SKIP_DIRS = {"competitions", "datasets", "rsna-knee-abnormality-detection"}
CKPT_KINDS = ("best", "swa")
FINGERPRINT_BYTES = 8 << 20


def our_weight_bases():
    """Lists the weight datasets checkpoints are searched in (S6 hygiene e).

    Returns:
        The dry run's weights directory, else every attached mount of WEIGHT_SLUGS, in
        WEIGHT_SLUGS order.

    Raises:
        SystemExit: None of our weight datasets is attached.
    """
    if S7["weights_dir"] is not None:
        return [S7["weights_dir"]]
    found = []
    for slug in WEIGHT_SLUGS:
        candidates = [Path(f"/kaggle/input/datasets/{WEIGHT_OWNER}/{slug}"),
                      Path(f"/kaggle/input/{slug}")]
        candidates += sorted(Path("/kaggle/input").glob(f"*/*/{slug}"))
        candidates += sorted(Path("/kaggle/input").glob(f"*/{slug}"))
        hit = next((c for c in candidates if c.is_dir()), None)
        print(f"weights {slug}: {hit if hit else 'not attached'}")
        if hit is not None:
            found.append(hit)
    if not found:
        raise SystemExit(f"none of {WEIGHT_SLUGS} is attached under /kaggle/input")
    return found


def fingerprint(path):
    """Hashes a file's size and its first FINGERPRINT_BYTES (runs of one recipe have equal sizes).

    Args:
        path: A checkpoint file.

    Returns:
        A hex digest.
    """
    h = hashlib.sha1(str(path.stat().st_size).encode())
    with open(path, "rb") as f:
        h.update(f.read(FINGERPRINT_BYTES))
    return h.hexdigest()


def find_checkpoints():
    """Finds every complete fold set in our weight datasets and lists their paths, fold-major.

    A set is the best_fold<k>.pt or swa_fold<k>.pt files of one directory. It is complete when
    it holds all N_FOLDS folds, or when its directory name ends in _full and it holds one file.
    Anything else is reported and skipped. A set whose files are identical to one already found
    (the same run in two datasets) is reported and skipped, not scored twice.

    Returns:
        Every set's fold 0, then every set's fold 1, and so on.

    Raises:
        SystemExit: No complete set was found.
    """
    t0 = time.time()
    sets = {}
    for base in our_weight_bases():
        for path in Path(base).rglob("*_fold*.pt"):
            kind, _, fold = path.stem.partition("_fold")
            if kind in CKPT_KINDS and fold.isdigit():
                sets.setdefault((path.parent, kind), {})[int(fold)] = path
    complete = {k: v for k, v in sets.items()
                if set(v) == set(range(N_FOLDS)) or (k[0].name.endswith("_full") and len(v) == 1)}
    for k in [k for k in complete if any(fnmatch.fnmatch(k[0].name, p) for p in EXCLUDE_SETS)]:
        print(f"excluded by EXCLUDE_SETS: {k[0].name} ({k[1]})")
        del complete[k]
    for (d, kind), v in sorted(sets.items()):
        if (d, kind) not in complete:
            print(f"skipping {d} ({kind}): folds {sorted(v)} of {N_FOLDS}")
    if not complete:
        raise SystemExit(f"no complete checkpoint set in {our_weight_bases()}")
    keys, seen = [], {}
    for key in sorted(complete, key=lambda k: (k[0].name, k[1], str(k[0]))):
        sig = (key[1], tuple(fingerprint(complete[key][f]) for f in sorted(complete[key])))
        if sig in seen:
            print(f"skipping {key[0]} ({key[1]}): same files as {seen[sig]}")
            continue
        seen[sig] = key[0]
        keys.append(key)
    for d, kind in keys:
        print(f"set: {d} ({kind}, {len(complete[(d, kind)])} checkpoint(s))")
    print(f"checkpoint search: {len(keys)} sets, {sum(len(complete[k]) for k in keys)} "
          f"checkpoints in {time.time() - t0:.1f} s")
    return [complete[key][fold] for fold in range(N_FOLDS) for key in keys if fold in complete[key]]


def pass_hint(member):
    """Pre-build guess of one member's seconds per study per TTA pass on 2 x T4.

    Args:
        member: A Member.

    Returns:
        PASS_HINT_S for its family's backbone word, else ConvNeXt's scaled by the pixel count.
    """
    word = re.match(r"[a-z]+", member.cfg["BACKBONE"].lower())
    word = word.group(0) if word else ""
    if word in PASS_HINT_S:
        return PASS_HINT_S[word]
    return PASS_HINT_S["convnext"] * (member.contract.side / 320) ** 2


def describe(members):
    """Prints one line per group and per family.

    Args:
        members: Members to describe.
    """
    for c, ms in group_members(members).items():
        sets = list(dict.fromkeys(m.set_name for m in ms))
        heads = sorted({m.cfg["HEAD"] for m in ms})
        backbones = sorted({m.cfg["BACKBONE"] for m in ms})
        print(f"  group {c.name}: {len(ms)} members | {backbones} | head {heads} | sets {sets}")
    fams = {}
    for m in members:
        fams.setdefault(m.family, set()).add(m.set_name)
    for f, s in fams.items():
        print(f"  family {f} (weight {FAMILY_WEIGHT.get(f, 1.0)}): {len(s)} sets "
              f"{sorted(s)}")

In [ ]:
# Describe every member, then drop sets whose cost hints cannot fit, before building any cache.
_s7_t = time.time()
try:
    S7["members"] = discover_members(find_checkpoints(), FAMILY)
    if not S7["members"]:
        raise RuntimeError("no checkpoint names an input this kernel can build")
    for _m in S7["members"]:
        if _m.cfg.get("_EMA"):
            S7.setdefault("ema_sets", set()).add(_m.set_name)
    print(f"{len(S7['members'])} members | EMA weights in {sorted(S7.get('ema_sets', []))}")
    describe(S7["members"])

    _studies = pd.read_csv(_s7_comp_root() / "sample_submission.csv")["StudyInstanceUID"]
    _n_plan = int(_override.get("plan_studies", 0) or 0) or len(_studies)
    _left = BUDGET_H * 3600 - (time.time() - NB_T0) - (PUBLIC_EST_H * 3600 if PUBLIC_STAGE else 0)
    _hint = {m.path: pass_hint(m) for m in S7["members"]}
    _build = {}
    for _c in {m.contract.cache for m in S7["members"]}:
        _build[_c.name] = BUILD_HINT_S[_c.rules] * _n_plan
    _kept, _offs, _log = trim_members(S7["members"], _hint, _n_plan, _left,
                                      len(S7["tta_offsets"]), PRIORITY, _build)
    _proj = projected_seconds(S7["members"], _hint, _n_plan, len(S7["tta_offsets"]), _build)
    print(f"pre-build plan for {_n_plan} studies: hinted {_proj / 3600:.2f} h "
          f"(caches {sum(_build.values()) / 3600:.2f} h) against {_left / 3600:.2f} h left")
    for _line in _log:
        if _line.startswith("dropped"):
            S7["members"] = _kept
            S7["dropped"].append(f"pre-build: {_line}")
            print(f"PRE-BUILD TRIM: {_line}")
    S7["caches"] = sorted({m.contract.cache for m in S7["members"]}, key=lambda c: c.name)
    print("caches to build:", [c.name for c in S7["caches"]])
except (Exception, SystemExit) as exc:
    _s7_ours_failed(exc)
S7["times"]["plan"] = time.time() - _s7_t

## Our stage, part 1: the test caches

The cache code below is `knee/cache.py`'s, copied verbatim: the slot builder `process_study` (v3 and v4 differ only in their `CacheConfig`: v4 ignores negated fat-suppression words and falls back to InstanceNumber order) and the v5 builder `process_study_v5`. Every cache reads fat suppression from the `Fat_Suppression` column, NaN-safe (S6 hygiene b).

One worker per study builds every needed cache in turn. While it works on a study, `read_header`, `order_files` and `read_pixels` are memoised, so a DICOM file that two caches sample is decoded once. The memo hands back exactly what the reader returned, so each cache is byte-identical to a separate build (checked on the Mac: 3 studies x 3 caches, every member equal).

In [ ]:
import multiprocessing as mp
import os
import random
import re
import time
import traceback
import zlib
from contextlib import contextmanager
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import pydicom

random.seed(42)
np.random.seed(42)
print("numpy", np.__version__, "| pydicom", pydicom.__version__)

# --- verbatim from src/knee/cache.py: DEFAULT_SLOTS, SLOT_SPEC, FALLBACK_ORDER, HDR_TAGS, ORDER_TAGS, CANON_SIGN, _RX_LOC, _RX_T1, _RX_T2, _RX_PD, _RX_STIR, _RX_FS, _RX_FS_V4, _RX_FS_NEG, CacheConfig, resize, to_float, read_header, image_centre_x, fs_in_description, classify_contrast, is_localizer, pick_slots, study_laterality, study_patient_id
DEFAULT_SLOTS = ("SAG_FS", "COR_FS", "AX_FS", "SAG_NOFS", "COR_T1", "SAG_T1")

SLOT_SPEC = {
    "SAG_FS":   ("Sagittal", True,  {"PD", "T2", "STIR", "UNK", "GRE"}),
    "COR_FS":   ("Coronal",  True,  {"PD", "T2", "STIR", "UNK", "GRE"}),
    "AX_FS":    ("Axial",    True,  {"PD", "T2", "STIR", "UNK", "GRE"}),
    "SAG_NOFS": ("Sagittal", False, {"PD", "T2", "GRE", "UNK"}),
    "COR_T1":   ("Coronal",  None,  {"T1"}),
    "SAG_T1":   ("Sagittal", None,  {"T1"}),
}

FALLBACK_ORDER = ["T2", "PD", "GRE", "UNK"]

HDR_TAGS = [
    "SeriesDescription", "ProtocolName", "SequenceName", "ScanningSequence", "ScanOptions",
    "RepetitionTime", "EchoTime", "InversionTime", "Laterality", "ImageLaterality",
    "PixelSpacing", "Rows", "Columns", "ImageOrientationPatient", "ImagePositionPatient",
    "PhotometricInterpretation", "SeriesInstanceUID", "PatientID",
]

ORDER_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]

CANON_SIGN = {0: 1.0, 1: 1.0, 2: -1.0}   # LPS: +x left, +y posterior, -z inferior (display order)

_RX_LOC = re.compile(r"loc|scout|survey|plan|calib|smartbrain|3.?plane", re.I)

_RX_T1 = re.compile(r"\bt1\b|t1w|t1_|_t1|t1 ", re.I)

_RX_T2 = re.compile(r"\bt2\b|t2w|t2_|_t2|t2 ", re.I)

_RX_PD = re.compile(r"\bpd\b|\bdp\b|proton|dens|pdw", re.I)

_RX_STIR = re.compile(r"stir|tirm", re.I)

_RX_FS = re.compile(r"\bfs\b|fat.?sat|spir|spair|fatsup|\bsat\b|fs_|_fs|fs ", re.I)

_RX_FS_V4 = re.compile(_RX_FS.pattern + r"|fat.?sup|stir|tirm", re.I)

_RX_FS_NEG = re.compile(r"(?<![a-z0-9])(?:non|no|not|without|w/o|w/out|wo)[\s_\-./]*$", re.I)

@dataclass(frozen=True)
class CacheConfig:
    """Everything the cache build needs beyond the input and output paths.

    Attributes:
        slots: Slot names in cache order; the training head indexes by position.
        n_slices: Slices stored per slot.
        band: (lo, hi) fractions of the ordered stack to sample between.
        img: Output side length in pixels. The builder takes any size, but
            knee.common.slice_descriptors (and its copy in the submission kernel) pools each
            slice on a 20 x 20 grid by reshape, so it needs a multiple of 20: 320 works,
            v4's 384 raises until that function pools on uneven blocks.
        crop_mm: Side of the physical square centre crop, in mm.
        clip_lo: Lower percentile of the per-series clip that maps to 0.
        clip_hi: Upper percentile of the per-series clip that maps to 255.
        min_slices: A series with fewer files is a localizer and never fills a slot.
        lat_dead_mm: |centre x| below this leaves laterality unknown, so nothing is mirrored.
        compress: Whether the npz is written compressed.
        fs_negation: Ignore a fat-suppression word in the description when a negation word
            ("non", "no", "not", "without", "w/o") sits right before it. Off in v3.
        instance_fallback: When under 80 % of a series' files carry geometry, order its
            slices by InstanceNumber (if at least 80 % carry one) before falling back to
            file name. Off in v3, which then keeps the file-name order.
        cache_key: Names the cache version. Any value but 'cache_v3' is also written into
            every npz as `cache_key`.
        layout: 'slots' writes the six-slot npz of v3/v4 (process_study); 'series' writes
            the v5 every-series npz (process_study_v5). `slots` then only names the slot
            each series would have taken.
        fs_column: The series-CSV column that forces fat suppression. v3 and v4 read
            Fluid_Sensitive; v5 reads Fat_Suppression. The two agree on every train and
            test row delivered so far, but only the second means what the flag is used for.
        n_shards: How many shards the corpus is split into (1 disables sharding).
        shard_index: Which shard this run builds.
        n_workers: Pool size; 0 means os.cpu_count().
        time_budget_h: Wall-clock hours after which the run stops, resumable.
        seed: Seed for the random and numpy generators.
    """

    slots: tuple[str, ...] = DEFAULT_SLOTS
    n_slices: int = 24
    band: tuple[float, float] = (0.10, 0.90)
    img: int = 320
    crop_mm: float = 140.0
    clip_lo: float = 0.5
    clip_hi: float = 99.5
    min_slices: int = 6
    lat_dead_mm: float = 20.0
    compress: bool = True
    fs_negation: bool = False
    instance_fallback: bool = False
    cache_key: str = "cache_v3"
    layout: str = "slots"
    fs_column: str = "Fluid_Sensitive"
    n_shards: int = 3
    shard_index: int = 0
    n_workers: int = 0
    time_budget_h: float = 11.0
    seed: int = 42

    @property
    def n_slots(self) -> int:
        """Number of series slots per study.

        Returns:
            The length of `slots`.
        """
        return len(self.slots)

    @classmethod
    def v4(cls, **overrides):
        """Builds the cache-v4 configuration.

        Args:
            **overrides: Fields set on top of the v4 settings, such as shard_index.

        Returns:
            A CacheConfig with img 384, fs_negation and instance_fallback on, and cache_key
            'cache_v4'.
        """
        settings = {"img": 384, "fs_negation": True, "instance_fallback": True,
                    "cache_key": "cache_v4"}
        return cls(**{**settings, **overrides})

    @classmethod
    def v5(cls, **overrides):
        """Builds the cache-v5 configuration.

        Args:
            **overrides: Fields set on top of the v5 settings, such as shard_index or img.

        Returns:
            A CacheConfig with the series layout, 32 slices over the (0.02, 0.98) band,
            img 288, the v4 header rules, fs_column 'Fat_Suppression', four shards and
            cache_key 'cache_v5'.
        """
        settings = {"layout": "series", "n_slices": 32, "band": (0.02, 0.98), "img": 288,
                    "fs_negation": True, "instance_fallback": True,
                    "fs_column": "Fat_Suppression", "cache_key": "cache_v5", "n_shards": 4}
        return cls(**{**settings, **overrides})

try:
    import cv2
    cv2.setNumThreads(0)

    def resize(a, size):
        """Resizes one slice to size x size with OpenCV area interpolation.

        Args:
            a: 2-D float array.
            size: Output side length in pixels.

        Returns:
            The resized slice.
        """
        return cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA)
except ImportError:
    from PIL import Image

    def resize(a, size):
        """Resizes one slice to size x size with PIL bilinear interpolation.

        Args:
            a: 2-D float array.
            size: Output side length in pixels.

        Returns:
            The resized slice.
        """
        return np.asarray(Image.fromarray(a).resize((size, size), Image.BILINEAR))

def to_float(x, default=None):
    """Coerces a DICOM value to float.

    Args:
        x: Any value, typically a pydicom element value.
        default: What to return when x is not numeric.

    Returns:
        The float, or default.
    """
    try:
        return float(x)
    except (TypeError, ValueError):
        return default

def read_header(series_path):
    """Parses one series' header into a plain dict. Never raises.

    The middle file of the series is read with stop_before_pixels, so one header read
    stands for the whole series.

    Args:
        series_path: Directory holding the series' .dcm files.

    Returns:
        A dict with n_files and hdr_ok always present, and, when the header could be read,
        desc, scan_seq, scan_opts, tr, te, ti, lat_tag, patient_id, ps_row, ps_col, rows,
        cols, iop, ipp and centre_x. hdr_ok is 0 when only the file count is known.
    """
    files = sorted(p for p in Path(series_path).iterdir() if p.name.lower().endswith(".dcm"))
    h = {"n_files": len(files), "hdr_ok": 0}
    if not files:
        return h
    try:
        ds = pydicom.dcmread(str(files[len(files) // 2]), stop_before_pixels=True,
                             force=True, specific_tags=HDR_TAGS)
    except (pydicom.errors.InvalidDicomError, OSError, ValueError, KeyError, AttributeError):
        return h
    h["hdr_ok"] = 1
    h["desc"] = " ".join(str(getattr(ds, t, "") or "") for t in
                         ("SeriesDescription", "ProtocolName", "SequenceName"))
    h["scan_seq"] = str(getattr(ds, "ScanningSequence", "") or "")
    h["scan_opts"] = str(getattr(ds, "ScanOptions", "") or "")
    h["tr"] = to_float(getattr(ds, "RepetitionTime", None))
    h["te"] = to_float(getattr(ds, "EchoTime", None))
    h["ti"] = to_float(getattr(ds, "InversionTime", None))
    lat = str(getattr(ds, "ImageLaterality", "")
              or getattr(ds, "Laterality", "") or "").strip().upper()
    h["lat_tag"] = lat if lat in ("L", "R") else ""
    h["patient_id"] = str(getattr(ds, "PatientID", "") or "").strip()
    ps = getattr(ds, "PixelSpacing", None)
    h["ps_row"] = to_float(ps[0]) if ps is not None and len(ps) >= 2 else None
    h["ps_col"] = to_float(ps[1]) if ps is not None and len(ps) >= 2 else None
    h["rows"] = int(getattr(ds, "Rows", 0) or 0)
    h["cols"] = int(getattr(ds, "Columns", 0) or 0)
    iop = getattr(ds, "ImageOrientationPatient", None)
    ipp = getattr(ds, "ImagePositionPatient", None)
    h["iop"] = [float(x) for x in iop[:6]] if iop is not None and len(iop) >= 6 else None
    h["ipp"] = [float(x) for x in ipp[:3]] if ipp is not None and len(ipp) >= 3 else None
    h["centre_x"] = image_centre_x(h)
    return h

def image_centre_x(h):
    """Patient-x (LPS: +x = patient's left) of the image centre, in mm.

    Args:
        h: A header dict from read_header; iop, ipp, ps_row/ps_col and rows/cols are read.

    Returns:
        The x coordinate in mm, or None when the geometry tags are missing.
    """
    if not h.get("iop") or not h.get("ipp") or not h.get("ps_row") or not h.get("rows"):
        return None
    r = np.asarray(h["iop"][:3])   # direction of increasing column index
    c = np.asarray(h["iop"][3:])   # direction of increasing row index
    centre = (np.asarray(h["ipp"]) + r * (h["cols"] - 1) / 2 * h["ps_col"]
              + c * (h["rows"] - 1) / 2 * h["ps_row"])
    return float(centre[0])

def fs_in_description(desc, ignore_negated=False):
    """Says whether a series description names fat suppression.

    Args:
        desc: SeriesDescription, ProtocolName and SequenceName joined by spaces.
        ignore_negated: Use the v4 test: the v3 words plus "fat sup(pressed)", STIR and TIRM,
            where a mention directly preceded by a negation word ("non fatsat", "no_fs",
            "w/o fs", "nonfs") does not count. False is the v3 test, where every mention
            counts.

    Returns:
        True when at least one mention counts.
    """
    if not ignore_negated:
        return bool(_RX_FS.search(desc))
    return any(not _RX_FS_NEG.search(desc[:m.start()]) for m in _RX_FS_V4.finditer(desc))

def classify_contrast(h, csv_fs, fs_negation=False):
    """Decides a series' tissue contrast and whether it is fat-suppressed.

    Precedence: SeriesDescription words, then TR/TE (TR < 800 ms -> T1; TE > 60 ms -> T2;
    else PD), with STIR counted as fat-suppressed whatever the other flags say.

    Args:
        h: A header dict from read_header.
        csv_fs: The delivered Fluid_Sensitive flag for this series; truthy forces fs.
        fs_negation: Whether a negated FS mention in the description is ignored (v4).

    Returns:
        The contrast, one of {T1, PD, T2, STIR, GRE, UNK}, and the fat-suppression flag.
    """
    desc = h.get("desc", "") or ""
    opts = {t.strip().upper() for t in re.split(r"[\\/,;]", h.get("scan_opts") or "")}
    fs = (bool(csv_fs) or fs_in_description(desc, fs_negation)
          or bool(opts & {"FS", "FAT_SAT", "FATSAT", "SPIR", "SPAIR"}))
    if _RX_STIR.search(desc) or (h.get("ti") and 80 <= h["ti"] <= 250):
        return "STIR", True
    if _RX_T1.search(desc):
        return "T1", fs
    if _RX_T2.search(desc):
        return "T2", fs
    if _RX_PD.search(desc):
        return "PD", fs
    tr, te = h.get("tr"), h.get("te")
    if tr is not None and te is not None:
        if tr < 800:
            return "T1", fs
        if te > 60:
            return "T2", fs
        return "PD", fs
    if "GR" in (h.get("scan_seq") or ""):
        return "GRE", fs
    return "UNK", fs

def is_localizer(h, min_slices=6):
    """Says whether a series is a localizer/scout rather than a diagnostic acquisition.

    Args:
        h: A header dict from read_header.
        min_slices: Fewest files a diagnostic acquisition may have.

    Returns:
        True when the series has fewer than min_slices files or its description names a
        localizer, survey, plan or calibration scan.
    """
    return h.get("n_files", 0) < min_slices or bool(_RX_LOC.search(h.get("desc", "") or ""))

def pick_slots(meta, slots=DEFAULT_SLOTS):
    """Chooses at most one series per cache slot.

    Pass 1 fills every slot from series that match its spec exactly. Pass 2 fills any FS slot
    still empty with a leftover same-plane non-FS T2, then PD (a minority of studies have no
    fat-suppressed sagittal at all; their T2 is fluid-sensitive and otherwise unused).
    A series never fills two slots. Candidates are ranked by slice count, UID as the
    tie-break, so the result does not depend on the order `meta` arrives in.

    Args:
        meta: One dict per series with uid, plane, fs, contrast, n_files and localizer.
        slots: Slot names in cache order.

    Returns:
        Two lists of one entry per slot, in `slots` order: the chosen series UID per slot
        (None where the slot stays empty), and whether that slot was filled by the pass-2
        fallback.
    """
    used, chosen, fallback = set(), [], [False] * len(slots)
    for name in slots:
        plane, want_fs, contrasts = SLOT_SPEC[name]
        cands = [m for m in meta
                 if m["uid"] not in used and not m["localizer"] and m["plane"] == plane
                 and m["contrast"] in contrasts
                 and (want_fs is None or bool(m["fs"]) == want_fs)]
        cands.sort(key=lambda m: (-m["n_files"], m["uid"]))
        pick = cands[0]["uid"] if cands else None
        if pick is not None:
            used.add(pick)
        chosen.append(pick)
    for i, name in enumerate(slots):
        plane, want_fs, _ = SLOT_SPEC[name]
        if chosen[i] is not None or want_fs is not True:
            continue
        for con in FALLBACK_ORDER:
            cands = [m for m in meta if m["uid"] not in used and not m["localizer"]
                     and m["plane"] == plane and m["contrast"] == con]
            if cands:
                cands.sort(key=lambda m: (-m["n_files"], m["uid"]))
                chosen[i] = cands[0]["uid"]
                used.add(chosen[i])
                fallback[i] = True
                break
    return chosen, fallback

def study_laterality(meta, lat_dead_mm=20.0):
    """Decides which knee a study images.

    The Laterality/ImageLaterality tag wins when any series carries one; otherwise the
    median image-centre x decides, unless it falls inside the dead zone around the midline.

    Args:
        meta: One dict per series, with lat_tag and centre_x.
        lat_dead_mm: Half-width of the dead zone around the midline, in mm.

    Returns:
        The side, one of {'L', 'R', 'U'}, and where it came from: 'tag', 'geom',
        'geom_dead:<x>' or 'none'.
    """
    tags = [m["lat_tag"] for m in meta if m.get("lat_tag")]
    if tags:
        side = max(("L", "R"), key=tags.count)
        return side, "tag"
    xs = [m["centre_x"] for m in meta if m.get("centre_x") is not None]
    if xs:
        x = float(np.median(xs))
        if abs(x) >= lat_dead_mm:
            return ("L" if x > 0 else "R"), "geom"
        return "U", f"geom_dead:{x:.0f}"
    return "U", "none"

def study_patient_id(meta):
    """Picks the study's PatientID from its series headers.

    Args:
        meta: One dict per series, with patient_id ('' when the tag was absent).

    Returns:
        The most frequent non-empty PatientID, the alphabetically first on a tie, or ''
        when no series carries one.
    """
    ids = [m["patient_id"] for m in meta if m.get("patient_id")]
    return max(sorted(set(ids)), key=ids.count) if ids else ""

In [ ]:
# --- verbatim from src/knee/cache.py: order_files, sample_indices, read_pixels, crop_mm, canonical_flips, load_series, out_path, series_metadata, build_volume, process_study
def order_files(series_path, instance_fallback=False, positions=None):
    """Sorts a series' files into through-plane order.

    Each file is keyed by projecting ImagePositionPatient onto the slice normal, falling
    back to InstanceNumber and then to the file's position in the directory listing. A
    part-geometric sort is worse than none, so when fewer than 80 % of files carry usable
    geometry v3 keeps the plain name order, which is random for SOP-UID file names. With
    instance_fallback the files are then sorted by InstanceNumber when at least 80 % carry
    one, and only otherwise left in name order.

    Args:
        series_path: Directory holding the series' .dcm files.
        instance_fallback: Try InstanceNumber order before name order (v4).
        positions: A list that, when given and the order is geometric, is extended with
            each ordered file's position along the slice normal in mm (v5 reads the slice
            spacing from it). None, the default, leaves the function as v3/v4 call it.

    Returns:
        The ordered paths; the slice normal as an unquantised vector, or None when the
        order is not geometric; and where the order came from: 'geom', 'instance' or
        'name' ('' when the directory holds no .dcm file).
    """
    files = sorted(p for p in Path(series_path).iterdir() if p.name.lower().endswith(".dcm"))
    if not files:
        return [], None, ""
    geom, inst, normal = [], [], None
    for f in files:
        g = k = None
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True, force=True,
                                 specific_tags=ORDER_TAGS)
            ipp = getattr(ds, "ImagePositionPatient", None)
            iop = getattr(ds, "ImageOrientationPatient", None)
            if ipp is not None and iop is not None and len(ipp) >= 3 and len(iop) >= 6:
                p = np.asarray([float(x) for x in ipp[:3]])
                o = np.asarray([float(x) for x in iop[:6]])
                n = np.cross(o[:3], o[3:])
                d = float(np.dot(p, n))
                if np.isfinite(d):
                    g, normal = d, n
            # v3 reads InstanceNumber only where geometry is missing; keeping that means a
            # malformed tag on a geometric file cannot change its v3 key.
            if g is None or instance_fallback:
                num = getattr(ds, "InstanceNumber", None)
                if num is not None:
                    k = float(num)
        except (pydicom.errors.InvalidDicomError, OSError, ValueError, KeyError,
                AttributeError, TypeError):
            pass
        geom.append(g)
        inst.append(k)
    n_geom = sum(g is not None for g in geom)
    n_inst = sum(k is not None for k in inst)
    if n_geom >= 0.8 * len(files) or (n_geom == 0 and not instance_fallback):
        keys = [g if g is not None else k if k is not None else float(i)
                for i, (g, k) in enumerate(zip(geom, inst))]
        order = [f for _, f in sorted(zip(keys, files), key=lambda t: t[0])]
        if positions is not None and n_geom:
            positions.extend(sorted(keys))
        return order, normal, "geom" if n_geom else ("instance" if n_inst else "name")
    if instance_fallback and n_inst >= 0.8 * len(files):
        idx = sorted(range(len(files)), key=lambda i: (inst[i] is None, inst[i] or 0.0, i))
        return [files[i] for i in idx], None, "instance"
    return files, None, "name"

def sample_indices(n, k, band):
    """Picks k evenly spaced slice indices from the central band of a stack.

    Args:
        n: How many slices the series has.
        k: How many to sample; indices repeat when k exceeds the band's width.
        band: (lo, hi) fractions of the ordered stack to sample between, inclusive.

    Returns:
        k integer indices, ascending, all within [0, n - 1]. All zeros when n <= 0.
    """
    if n <= 0:
        return np.zeros(k, dtype=int)
    lo, hi = band
    return np.linspace(lo * (n - 1), hi * (n - 1), k).round().astype(int)

def read_pixels(path):
    """Reads one DICOM file's pixels as a rescaled 2-D float array.

    Args:
        path: The .dcm file.

    Returns:
        The slice after RescaleSlope/Intercept, negated when the photometric
        interpretation is MONOCHROME1 so that bright always means high signal, or None
        when the file could not be decoded.
    """
    try:
        ds = pydicom.dcmread(str(path), force=True)
        a = ds.pixel_array.astype(np.float32)
        if a.ndim == 3:
            a = a[0] if a.shape[0] < a.shape[-1] else a[..., 0]
        a = (a * float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
             + float(getattr(ds, "RescaleIntercept", 0.0) or 0.0))
        if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
            a = -a
        return a
    # Pixel decoders raise a wide range of types (unsupported transfer syntax, truncated
    # file, missing element). One bad slice must never fail the study, so the list is broad,
    # but it is still narrower than `except Exception`.
    except (pydicom.errors.InvalidDicomError, OSError, ValueError, KeyError, AttributeError,
            TypeError, NotImplementedError, RuntimeError):
        return None

def crop_mm(a, ps_row, ps_col, size_mm):
    """Crops a slice to a fixed physical square around its centre.

    One output pixel is then the same physical size for every series. When the field of
    view is smaller than the requested square, the margin is padded with the slice's own
    minimum value (not zero), so the pad matches the image's background level.

    Args:
        a: The 2-D slice.
        ps_row: Row pixel spacing in mm; a falsy value returns `a` unchanged.
        ps_col: Column pixel spacing in mm; a falsy value returns `a` unchanged.
        size_mm: Side of the square crop in mm.

    Returns:
        A float32 array of round(size_mm / ps_row) x round(size_mm / ps_col) pixels, or
        `a` itself when the pixel spacing is unknown.
    """
    if not ps_row or not ps_col:
        return a
    hr, hc = int(round(size_mm / ps_row)), int(round(size_mm / ps_col))
    n_rows, n_cols = a.shape
    r0, c0 = (n_rows - hr) // 2, (n_cols - hc) // 2
    out = np.full((hr, hc), float(a.min()), dtype=np.float32)
    sr0, sc0 = max(r0, 0), max(c0, 0)
    sr1, sc1 = min(r0 + hr, n_rows), min(c0 + hc, n_cols)
    out[sr0 - r0: sr1 - r0, sc0 - c0: sc1 - c0] = a[sr0:sr1, sc0:sc1]
    return out

def canonical_flips(iop, normal, side):
    """Lists which axes of (slice, row, col) to flip for canonical orientation.

    Canonical is the standard radiological display: columns run right->left or
    anterior->posterior, rows run superior->inferior or anterior->posterior, and slices
    ascend toward +x/+y/-z. A right knee is then mirrored along whichever axis is aligned
    with patient x, so medial and lateral mean the same image direction for every study.

    Args:
        iop: ImageOrientationPatient as six floats (column direction then row direction),
            or None when the tag is missing.
        normal: The slice normal, or None when no geometry was recovered.
        side: 'L', 'R' or 'U' from study_laterality.

    Returns:
        The axes to flip, a subset of {0: slice, 1: row, 2: col}. Empty when iop is None.
    """
    if iop is None:
        return []
    col_dir = np.asarray(iop[:3])   # increasing column index
    row_dir = np.asarray(iop[3:])   # increasing row index
    dirs = {0: normal, 1: row_dir, 2: col_dir}
    flips = []
    for ax, d in dirs.items():
        if d is None:
            continue
        dom = int(np.argmax(np.abs(d)))
        if d[dom] * CANON_SIGN[dom] < 0:
            flips.append(ax)
        if side == "R" and dom == 0:        # x-aligned axis: mirror right knees
            flips = [f for f in flips if f != ax] if ax in flips else flips + [ax]
    return flips

def load_series(series_path, h, side, n_slices, img, band, crop, clip_lo, clip_hi,
                instance_fallback=False):
    """Loads one series into the cache's uint8 slot format.

    Slices are ordered, sampled over the central band, physically cropped, resized,
    flipped into canonical orientation and finally scaled by a percentile clip taken over
    the whole sampled stack. A slice that fails to decode is replaced by the previous good
    one, so the slot always has n_slices planes.

    Args:
        series_path: Directory holding the series' .dcm files.
        h: The series' header dict, read for pixel spacing and orientation.
        side: The study's laterality, for the right-knee mirror.
        n_slices: Slices to store.
        img: Output side length in pixels.
        band: (lo, hi) fractions of the stack to sample between.
        crop: Physical crop size in mm.
        clip_lo: Lower percentile of the clip, over the whole sampled stack.
        clip_hi: Upper percentile of the clip.
        instance_fallback: Passed to order_files.

    Returns:
        A (n_slices, img, img) uint8 array, or None when the series yielded nothing; a
        note, one of 'ok', 'partial:<n>' with the count of substituted slices, 'no_dcm' or
        'all_slices_failed'; and the slice-order source from order_files.
    """
    files, normal, order_src = order_files(series_path, instance_fallback)
    if not files:
        return None, "no_dcm", order_src
    idx = sample_indices(len(files), n_slices, band)
    planes, bad = [], 0
    for i in idx:
        a = read_pixels(files[int(i)])
        if a is None:
            bad += 1
            planes.append(None)
            continue
        cropped = crop_mm(a, h.get("ps_row"), h.get("ps_col"), crop)
        planes.append(resize(cropped, img).astype(np.float32))
    good = [p for p in planes if p is not None]
    if not good:
        return None, "all_slices_failed", order_src
    last = None
    for j, p in enumerate(planes):
        if p is None:
            planes[j] = last if last is not None else good[0]
        else:
            last = p
    vol = np.stack(planes, axis=0)
    for ax in canonical_flips(h.get("iop"), normal, side):
        vol = np.flip(vol, axis=ax)
    lo, hi = np.percentile(vol, (clip_lo, clip_hi))
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    return (np.ascontiguousarray((vol * 255.0).round().astype(np.uint8)),
            f"partial:{bad}" if bad else "ok", order_src)

def out_path(study, out_dir):
    """Gives the cache file path for one study.

    Args:
        study: StudyInstanceUID.
        out_dir: Directory the cache is written to.

    Returns:
        The .npz path under the output directory.
    """
    return Path(out_dir) / f"{study}.npz"

def series_metadata(study, rows, series_dir, cfg):
    """Reads every series of one study and classifies it.

    Args:
        study: StudyInstanceUID.
        rows: The study's rows from {split}_series.csv, as dicts.
        series_dir: Root of the {split}_series tree.
        cfg: The CacheConfig for this run.

    Returns:
        The per-series dicts slot selection needs (each carrying its header under 'h'),
        and the flat metadata rows written to the series-metadata CSV.
    """
    meta, meta_rows = [], []
    for r in rows:
        uid = str(r["SeriesInstanceUID"])
        path = Path(series_dir) / str(study) / uid
        h = read_header(path) if path.is_dir() else {"n_files": 0, "hdr_ok": 0}
        csv_fs = r.get(cfg.fs_column, 0)
        contrast, fs = classify_contrast(h, csv_fs, cfg.fs_negation)
        m = {"uid": uid, "plane": str(r.get("Anatomical_Plane", "")), "fs": fs,
             "contrast": contrast, "n_files": h.get("n_files", 0),
             "localizer": is_localizer(h, cfg.min_slices), "lat_tag": h.get("lat_tag", ""),
             "centre_x": h.get("centre_x"), "patient_id": h.get("patient_id", ""), "h": h}
        meta.append(m)
        meta_rows.append({"StudyInstanceUID": study, "SeriesInstanceUID": uid,
                          "plane": m["plane"],
                          "csv_fs": int(csv_fs or 0),
                          "fs": int(fs), "contrast": contrast,
                          "n_files": m["n_files"], "localizer": int(m["localizer"]),
                          "tr": h.get("tr"), "te": h.get("te"), "lat_tag": m["lat_tag"],
                          "centre_x": m["centre_x"], "ps_row": h.get("ps_row"),
                          "patient_id": m["patient_id"], "desc": (h.get("desc") or "")[:80]})
    return meta, meta_rows

def build_volume(study, picks, by_uid, series_dir, side, cfg):
    """Loads the chosen series into the study's slot array.

    One unreadable series costs its slot, not the study: the failure is recorded as a note
    and the slot stays zeros with present = 0.

    Args:
        study: StudyInstanceUID.
        picks: The chosen series UID per slot, None where the slot stays empty.
        by_uid: Series metadata dicts keyed by UID, each carrying its header under 'h'.
        series_dir: Root of the {split}_series tree.
        side: The study's laterality, for the right-knee mirror.
        cfg: The CacheConfig for this run.

    Returns:
        The (n_slots, n_slices, img, img) uint8 array, the uint8 presence mask, the per-slot
        notes as '<slot index>:<note>' strings, and the slice-order source per slot ('geom',
        'instance' or 'name'; '' where the slot is empty).
    """
    arr = np.zeros((cfg.n_slots, cfg.n_slices, cfg.img, cfg.img), dtype=np.uint8)
    present = np.zeros(cfg.n_slots, dtype=np.uint8)
    notes, order_src = [], [""] * cfg.n_slots
    for s, uid in enumerate(picks):
        if uid is None:
            continue
        path = Path(series_dir) / str(study) / uid
        try:
            vol, note, src = load_series(path, by_uid[uid]["h"], side, cfg.n_slices, cfg.img,
                                         cfg.band, cfg.crop_mm, cfg.clip_lo, cfg.clip_hi,
                                         cfg.instance_fallback)
        # Deliberate isolation point: one unreadable series must cost its slot, not the
        # study. The exception type is recorded in the manifest note.
        except Exception as e:
            vol, note = None, f"err:{type(e).__name__}"
        if vol is None:
            notes.append(f"{s}:{note}")
            continue
        arr[s] = vol
        present[s] = 1
        order_src[s] = src
        if note != "ok":
            notes.append(f"{s}:{note}")
    return arr, present, notes, order_src

def process_study(study, study_rows, series_dir, out_dir, cfg=CacheConfig()):
    """Builds one study's cache file. Never raises.

    The npz is the contract training and inference read. It holds:
      vol: uint8 (n_slots, n_slices, img, img), the slot volumes in cfg.slots order, zeros
        where the slot is empty;
      present: uint8 (n_slots,), 1 where the slot was filled;
      side: 0-d string array, the study's laterality ('L', 'R' or 'U');
      slot_uids: string (n_slots,), the SeriesInstanceUID behind each slot, '' when empty;
      slot_fallback: uint8 (n_slots,), 1 where an FS slot was filled by a non-FS series;
      cache_key: 0-d string array naming the cache version, e.g. 'cache_v4'. Absent from
        v3 files, whose layout is unchanged.

    Written atomically: the array goes to a .tmp.npz that is renamed into place, so an
    interrupted shard leaves no half-written file for the resume pass to trust.

    Args:
        study: StudyInstanceUID.
        study_rows: Every study's series rows keyed by StudyInstanceUID, as dicts.
        series_dir: Root of the {split}_series tree.
        out_dir: Directory the cache is written to.
        cfg: The CacheConfig for this run.

    Returns:
        A manifest record and one metadata row per series. The record's status is 'ok',
        'skip' when the file already exists, 'empty' when no slot could be filled, or
        'fail' with the traceback in its note. Its order_src holds the slice-order source
        per slot joined by '|' ('geom', 'instance' or 'name'; empty for an empty slot), and
        its patient_id the study's PatientID ('' when absent).
    """
    dst = out_path(study, out_dir)
    rec = {"StudyInstanceUID": study, "status": "ok", "slots": "", "side": "",
           "side_src": "", "fallback": "", "order_src": "", "patient_id": "", "note": ""}
    meta_rows = []
    try:
        if dst.is_file() and dst.stat().st_size > 0:
            rec["status"] = "skip"
            return rec, meta_rows

        meta, meta_rows = series_metadata(study, study_rows.get(study, []), series_dir, cfg)
        rec["patient_id"] = study_patient_id(meta)
        side, side_src = study_laterality(meta, cfg.lat_dead_mm)
        picks, fallback = pick_slots(meta, cfg.slots)
        by_uid = {m["uid"]: m for m in meta}
        arr, present, notes, order_src = build_volume(study, picks, by_uid, series_dir, side,
                                                      cfg)

        if present.sum() == 0:
            rec.update(status="empty", note=";".join(notes))
            return rec, meta_rows

        tmp = dst.with_suffix(".tmp.npz")
        save = np.savez_compressed if cfg.compress else np.savez
        key = {} if cfg.cache_key == "cache_v3" else {"cache_key": np.array(cfg.cache_key)}
        save(tmp, vol=arr, present=present, side=np.array(side),
             slot_uids=np.array([u or "" for u in picks]),
             slot_fallback=np.array(fallback, dtype=np.uint8), **key)
        os.replace(tmp, dst)
        rec.update(slots="".join(str(int(x)) for x in present), side=side, side_src=side_src,
                   fallback="".join(str(int(x)) for x in fallback),
                   order_src="|".join(order_src), note=";".join(notes))
        return rec, meta_rows
    # Deliberate isolation point: this is the whole body of a pool worker, and a shard of
    # 1500 studies must not die on one of them. The traceback goes into the manifest.
    except Exception:
        rec.update(status="fail", note=traceback.format_exc(limit=2).replace("\n", " | "))
        return rec, meta_rows

In [ ]:
# --- verbatim from src/knee/cache.py: PLANES_V5, CONTRASTS_V5, TYPE_ORDER_V5, N_META_V5, EXTRA_TAGS, read_series_extras, series_sort_key, load_series_v5, _type_code, process_study_v5
PLANES_V5 = ("Sagittal", "Coronal", "Axial")

CONTRASTS_V5 = ("T1", "PD", "T2", "STIR", "GRE", "UNK")

TYPE_ORDER_V5 = ((0, 1), (1, 1), (2, 1), (0, 0), (1, 0), (2, 0))

N_META_V5 = len(PLANES_V5) + 1 + len(CONTRASTS_V5)

EXTRA_TAGS = ["SeriesNumber", "SpacingBetweenSlices", "SliceThickness"]

def read_series_extras(series_path):
    """Reads the tags v5 stores beyond read_header's from the series' middle file.

    Args:
        series_path: Directory holding the series' .dcm files.

    Returns:
        A dict with series_number (-1 when absent), spacing_tag and thickness (None when
        absent). Never raises.
    """
    out = {"series_number": -1, "spacing_tag": None, "thickness": None}
    files = sorted(p for p in Path(series_path).iterdir() if p.name.lower().endswith(".dcm"))
    if not files:
        return out
    try:
        ds = pydicom.dcmread(str(files[len(files) // 2]), stop_before_pixels=True,
                             force=True, specific_tags=EXTRA_TAGS)
    except (pydicom.errors.InvalidDicomError, OSError, ValueError, KeyError, AttributeError):
        return out
    num = to_float(getattr(ds, "SeriesNumber", None))
    out["series_number"] = int(num) if num is not None else -1
    out["spacing_tag"] = to_float(getattr(ds, "SpacingBetweenSlices", None))
    out["thickness"] = to_float(getattr(ds, "SliceThickness", None))
    return out

def series_sort_key(m, slot, n_slots=len(DEFAULT_SLOTS)):
    """Gives the stable key that orders a study's series inside the v5 npz.

    Series that take a v3 slot come first, in slot order; then the rest by plane
    (sagittal, coronal, axial), FS before non-FS, contrast in CONTRASTS_V5 order, most
    files first and UID as the final tie-break. The order never depends on the order the
    series CSV lists them in.

    Args:
        m: A series dict from series_metadata (plane, fs, contrast, n_files, uid).
        slot: The v3-style slot the series takes under pick_slots, or -1.
        n_slots: Number of slots, so that "no slot" sorts after every slot.

    Returns:
        A tuple usable as a sort key.
    """
    plane = PLANES_V5.index(m["plane"]) if m["plane"] in PLANES_V5 else len(PLANES_V5)
    contrast = (CONTRASTS_V5.index(m["contrast"]) if m["contrast"] in CONTRASTS_V5
                else len(CONTRASTS_V5))
    return (slot if slot >= 0 else n_slots, plane, 0 if m["fs"] else 1, contrast,
            -int(m["n_files"]), m["uid"])

def load_series_v5(series_path, h, side, cfg):
    """Loads one series into the v5 format: the whole stack, with its geometry facts.

    Same steps as load_series (order, sample over the band, crop, resize, canonical flips,
    one percentile clip over the sampled stack, a failed slice replaced by the previous
    good one), plus the native slice count, the slice spacing and each stored slice's
    position in the stack.

    Args:
        series_path: Directory holding the series' .dcm files.
        h: The series' header dict, read for pixel spacing and orientation.
        side: The study's laterality, for the right-knee mirror.
        cfg: The CacheConfig (v5); reads n_slices, img, band, crop_mm, clip_lo, clip_hi
            and instance_fallback.

    Returns:
        A (n_slices, img, img) uint8 array, or None when the series yielded nothing; a note
        as load_series gives it; the slice-order source; and a dict with n_native,
        spacing_mm (the median gap between geometric slice positions, NaN when the order
        is not geometric) and slice_pos, a float32 (n_slices,) array giving each stored
        slice's position as a fraction of the stack, measured in the stored (canonical)
        direction so that it ascends.
    """
    positions = []
    files, normal, order_src = order_files(series_path, cfg.instance_fallback, positions)
    info = {"n_native": len(files), "spacing_mm": float("nan"),
            "slice_pos": np.zeros(cfg.n_slices, np.float32)}
    if not files:
        return None, "no_dcm", order_src, info
    n = len(files)
    if len(positions) > 1:
        gaps = np.abs(np.diff(np.asarray(positions, dtype=np.float64)))
        info["spacing_mm"] = float(np.median(gaps))
    idx = sample_indices(n, cfg.n_slices, cfg.band)
    planes, bad = [], 0
    for i in idx:
        a = read_pixels(files[int(i)])
        if a is None:
            bad += 1
            planes.append(None)
            continue
        cropped = crop_mm(a, h.get("ps_row"), h.get("ps_col"), cfg.crop_mm)
        planes.append(resize(cropped, cfg.img).astype(np.float32))
    good = [p for p in planes if p is not None]
    if not good:
        return None, "all_slices_failed", order_src, info
    last = None
    for j, p in enumerate(planes):
        if p is None:
            planes[j] = last if last is not None else good[0]
        else:
            last = p
    vol = np.stack(planes, axis=0)
    flips = canonical_flips(h.get("iop"), normal, side)
    for ax in flips:
        vol = np.flip(vol, axis=ax)
    q = idx / max(n - 1, 1)
    info["slice_pos"] = (1.0 - q[::-1] if 0 in flips else q).astype(np.float32)
    lo, hi = np.percentile(vol, (cfg.clip_lo, cfg.clip_hi))
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    return (np.ascontiguousarray((vol * 255.0).round().astype(np.uint8)),
            f"partial:{bad}" if bad else "ok", order_src, info)

def _type_code(m):
    """Short code for a series' type in the manifest, e.g. 'SF-PD' or 'AN-T1'.

    Args:
        m: A series dict with plane, fs and contrast.

    Returns:
        Plane initial, F or N for fat suppression, a dash and the contrast.
    """
    return f"{(m['plane'] or '?')[0]}{'F' if m['fs'] else 'N'}-{m['contrast']}"

def process_study_v5(study, study_rows, series_dir, out_dir, cfg=None):
    """Builds one study's cache-v5 file. Never raises.

    Every non-localizer series with at least one file is loaded; one that fails costs
    itself, not the study. The npz holds, for N stored series in series_sort_key order:
      vol_00 ... vol_{N-1}: uint8 (n_slices, img, img), one member per series, so a reader
        decompresses only the series it uses;
      n_series: 0-d int, N;
      uids: str (N,), SeriesInstanceUID;
      plane: int8 (N,), index into PLANES_V5, -1 when the CSV plane is unknown;
      fs: uint8 (N,), the v4 rules (negation-aware description, ScanOptions, STIR) or the
        Fat_Suppression column;
      contrast: int8 (N,), index into CONTRASTS_V5;
      n_native: int16 (N,), files in the series;
      spacing_mm: float32 (N,), median geometric slice gap, else SpacingBetweenSlices,
        else SliceThickness, else NaN;
      slot: int8 (N,), the v3 slot (DEFAULT_SLOTS index) pick_slots gives the series
        under these rules, -1 for none;
      slot_fallback: uint8 (N,), 1 where that slot is an FS slot filled by a non-FS series;
      series_number: int32 (N,), the DICOM SeriesNumber, -1 when absent;
      slice_pos: float16 (N, n_slices), each stored slice's fraction of the stack;
      order_src: str (N,), 'geom', 'instance' or 'name';
      side: 0-d str, 'L', 'R' or 'U'; cache_key: 0-d str, e.g. 'cache_v5'.

    Args:
        study: StudyInstanceUID.
        study_rows: Every study's series rows keyed by StudyInstanceUID, as dicts.
        series_dir: Root of the {split}_series tree.
        out_dir: Directory the cache is written to.
        cfg: The CacheConfig; None means CacheConfig.v5().

    Returns:
        A manifest record and one metadata row per listed series. The record carries the
        status ('ok', 'skip', 'empty' or 'fail'), n_series stored, n_failed, the stored
        types joined by '|', the six-character v3 slot presence, side, side_src, order_src
        per stored series, patient_id and notes. Metadata rows gain v5_index (position in
        the npz, -1 when not stored), slot, spacing_mm and series_number.
    """
    cfg = cfg or CacheConfig.v5()
    dst = out_path(study, out_dir)
    rec = {"StudyInstanceUID": study, "status": "ok", "n_series": 0, "n_failed": 0,
           "types": "", "slots": "", "side": "", "side_src": "", "order_src": "",
           "patient_id": "", "note": ""}
    meta_rows = []
    try:
        if dst.is_file() and dst.stat().st_size > 0:
            rec["status"] = "skip"
            return rec, meta_rows

        meta, meta_rows = series_metadata(study, study_rows.get(study, []), series_dir, cfg)
        rec["patient_id"] = study_patient_id(meta)
        side, side_src = study_laterality(meta, cfg.lat_dead_mm)
        picks, fallback = pick_slots(meta, cfg.slots)
        slot_of = {u: i for i, u in enumerate(picks) if u is not None}
        cands = [m for m in meta if not m["localizer"] and m["n_files"] > 0]
        cands.sort(key=lambda m: series_sort_key(m, slot_of.get(m["uid"], -1), cfg.n_slots))
        row_of = {r["SeriesInstanceUID"]: r for r in meta_rows}
        for r in meta_rows:
            r.update(v5_index=-1, slot=slot_of.get(r["SeriesInstanceUID"], -1),
                     spacing_mm=None, series_number=-1)

        arrays, kept, notes, n_failed = {}, [], [], 0
        for m in cands:
            path = Path(series_dir) / str(study) / m["uid"]
            try:
                vol, note, src, info = load_series_v5(path, m["h"], side, cfg)
                extras = read_series_extras(path)
            # Deliberate isolation point: one unreadable series must cost itself, not the
            # study. The exception type is recorded in the manifest note.
            except Exception as e:
                vol, note = None, f"err:{type(e).__name__}"
            if vol is None:
                n_failed += 1
                notes.append(f"{m['uid'][-8:]}:{note}")
                continue
            spacing = info["spacing_mm"]
            if not np.isfinite(spacing):
                spacing = extras["spacing_tag"] or extras["thickness"] or float("nan")
            k = len(kept)
            arrays[f"vol_{k:02d}"] = vol
            kept.append({"m": m, "src": src, "info": info, "spacing": spacing,
                         "series_number": extras["series_number"]})
            row_of[m["uid"]].update(v5_index=k, spacing_mm=spacing,
                                    series_number=extras["series_number"])
            if note != "ok":
                notes.append(f"{k}:{note}")

        rec["n_failed"] = n_failed
        if not kept:
            rec.update(status="empty", note=";".join(notes))
            return rec, meta_rows

        ms = [e["m"] for e in kept]
        stored = {m["uid"] for m in ms}
        slot_ids = [slot_of.get(m["uid"], -1) for m in ms]
        table = {
            "n_series": np.array(len(kept)),
            "uids": np.array([m["uid"] for m in ms]),
            "plane": np.array([PLANES_V5.index(m["plane"]) if m["plane"] in PLANES_V5
                               else -1 for m in ms], dtype=np.int8),
            "fs": np.array([int(bool(m["fs"])) for m in ms], dtype=np.uint8),
            "contrast": np.array([CONTRASTS_V5.index(m["contrast"]) for m in ms],
                                 dtype=np.int8),
            "n_native": np.array([e["info"]["n_native"] for e in kept], dtype=np.int16),
            "spacing_mm": np.array([e["spacing"] for e in kept], dtype=np.float32),
            "slot": np.array(slot_ids, dtype=np.int8),
            "slot_fallback": np.array([int(fallback[s]) if s >= 0 else 0 for s in slot_ids],
                                      dtype=np.uint8),
            "series_number": np.array([e["series_number"] for e in kept], dtype=np.int32),
            "slice_pos": np.stack([e["info"]["slice_pos"] for e in kept]).astype(np.float16),
            "order_src": np.array([e["src"] for e in kept]),
            "side": np.array(side),
            "cache_key": np.array(cfg.cache_key),
        }
        tmp = dst.with_suffix(".tmp.npz")
        save = np.savez_compressed if cfg.compress else np.savez
        save(tmp, **arrays, **table)
        os.replace(tmp, dst)
        rec.update(n_series=len(kept), types="|".join(_type_code(m) for m in ms),
                   slots="".join("1" if u in stored else "0" for u in picks),
                   side=side, side_src=side_src,
                   order_src="|".join(e["src"] for e in kept), note=";".join(notes))
        return rec, meta_rows
    # Deliberate isolation point: this is the whole body of a pool worker, and a shard of
    # 1100 studies must not die on one of them. The traceback goes into the manifest.
    except Exception:
        rec.update(status="fail", note=traceback.format_exc(limit=2).replace("\n", " | "))
        return rec, meta_rows

In [ ]:
# The joint build (knee/infer/inline_cache.py, with shared_reads acting on this notebook's
# globals instead of the knee.cache module).
FS_COLUMN = "Fat_Suppression"


def csv_flag(value):
    """Reads a 0/1 flag from a series-CSV value; missing, NaN or non-numeric reads 0.

    Args:
        value: The raw cell value.

    Returns:
        1 when the value is a finite non-zero number, else 0.
    """
    try:
        v = float(value)
    except (TypeError, ValueError):
        return 0
    return int(v != 0) if np.isfinite(v) else 0


def clean_rows(study_rows, column=FS_COLUMN):
    """Copies the series rows with the fat-suppression column made a clean 0/1 int.

    Args:
        study_rows: StudyInstanceUID -> list of series-row dicts.
        column: The column to read; Fluid_Sensitive is used when no row carries it.

    Returns:
        The cleaned rows and the column actually used.
    """
    has = any(column in r for rows in study_rows.values() for r in rows)
    used = column if has else "Fluid_Sensitive"
    out = {s: [{**r, used: csv_flag(r.get(used))} for r in rows]
           for s, rows in study_rows.items()}
    return out, used


def contract_config(contract, fs_column=FS_COLUMN, **overrides):
    """The CacheConfig that builds one cache contract.

    Args:
        contract: Cache version, side and slice count.
        fs_column: The series-CSV column read as the fat-suppression flag.
        **overrides: Further CacheConfig fields.

    Returns:
        A one-shard CacheConfig with the version's rules at the contract's side.
    """
    rules = CACHE_RULES[contract.rules]
    return CacheConfig(layout=rules["layout"], band=tuple(rules["band"]),
                       fs_negation=rules["fs_negation"],
                       instance_fallback=rules["instance_fallback"], img=contract.img,
                       n_slices=contract.n_slices, cache_key=f"cache_{contract.rules}",
                       fs_column=fs_column, n_shards=1, shard_index=0, **overrides)


@contextmanager
def shared_reads(ns):
    """Memoises the three DICOM readers of a builder namespace for one study.

    Args:
        ns: The namespace whose read_header, order_files and read_pixels the builders look up
            at call time (here, this notebook's globals()).

    Yields:
        The memo dict.
    """
    names = ("read_header", "order_files", "read_pixels")
    orig = {k: ns[k] for k in names}
    memo = {}

    def read_header(series_path):
        """Memoised read_header; returns a fresh copy of the header dict."""
        key = ("h", str(series_path))
        if key not in memo:
            memo[key] = orig["read_header"](series_path)
        return dict(memo[key])

    def order_files(series_path, instance_fallback=False, positions=None):
        """Memoised order_files, per (series, instance_fallback)."""
        key = ("o", str(series_path), bool(instance_fallback))
        if key not in memo:
            pos = []
            memo[key] = (orig["order_files"](series_path, instance_fallback, pos), pos)
        (files, normal, src), pos = memo[key]
        if positions is not None:
            positions.extend(pos)
        return list(files), normal, src

    def read_pixels(path):
        """Memoised read_pixels; the callers only read the array."""
        key = ("p", str(path))
        if key not in memo:
            memo[key] = orig["read_pixels"](path)
        return memo[key]

    ns.update(read_header=read_header, order_files=order_files, read_pixels=read_pixels)
    try:
        yield memo
    finally:
        ns.update(orig)


@dataclass
class JointWorker:
    """Picklable worker that builds every needed cache of one study with shared reads.

    Attributes:
        study_rows: Cleaned series rows keyed by StudyInstanceUID.
        series_dir: Root of the test_series tree.
        out_dirs: Cache name -> output directory.
        cfgs: Cache name -> CacheConfig.
    """

    study_rows: dict
    series_dir: str
    out_dirs: dict
    cfgs: dict = field(default_factory=dict)

    def __call__(self, study):
        """Builds one study in every cache.

        Args:
            study: StudyInstanceUID.

        Returns:
            The study and a dict cache name -> manifest record (with 'seconds' added).
        """
        recs = {}
        with shared_reads(globals()):
            for name, cfg in self.cfgs.items():
                t0 = time.time()
                build = process_study_v5 if cfg.layout == "series" else process_study
                rec, _ = build(study, self.study_rows, self.series_dir, self.out_dirs[name], cfg)
                rec["seconds"] = time.time() - t0
                recs[name] = rec
        return study, recs


def s7_find_root():
    """Finds the competition mount holding test.csv and test_series/.

    Returns:
        The mount, or None in a dry run without a fake mount.
    """
    if S7["dry_run"]:
        c = S7["comp_dir"]
        return c if (c / "test.csv").is_file() and (c / "test_series").is_dir() else None
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    for d1 in sorted(p for p in Path("/kaggle/input").iterdir() if p.is_dir()):
        for c in [d1] + sorted(p for p in d1.iterdir() if p.is_dir()):
            if (c / "test.csv").is_file() and (c / "test_series").is_dir():
                return c
    raise FileNotFoundError("competition mount with test.csv and test_series/ not found")

In [ ]:
# Build every needed cache in one pass over the test studies.
_s7_t = time.time()
if S7["ours_active"]:
    try:
        S7["cache_dirs"] = {c.name: S7["cache_root"] / c.name for c in S7["caches"]}
        _root = s7_find_root()
        if _root is None:
            # Dry run with pre-built caches: work_dir/s7_cache/<name>, or work_dir/cache (v3-320).
            for _name, _dir in S7["cache_dirs"].items():
                if not _dir.is_dir() and _name == "v3-320" and (S7["work_dir"] / "cache").is_dir():
                    S7["cache_dirs"][_name] = S7["work_dir"] / "cache"
                print(f"dry run: reading pre-built {_name} from {S7['cache_dirs'][_name]}")
        else:
            S7["comp_root"] = S7["comp_root"] or _root
            _series = pd.read_csv(_root / "test_series.csv")
            _rows = {s: g.to_dict("records") for s, g in _series.groupby("StudyInstanceUID")}
            _rows, _fs_used = clean_rows(_rows)
            print(f"root {_root} | series rows {len(_series)} | studies {len(_rows)} | "
                  f"fat suppression from {_fs_used}")
            if "Fat_Suppression" in _series and "Fluid_Sensitive" in _series:
                _a = pd.to_numeric(_series["Fat_Suppression"], errors="coerce")
                _b = pd.to_numeric(_series["Fluid_Sensitive"], errors="coerce")
                print(f"test series where Fat_Suppression != Fluid_Sensitive: "
                      f"{int((_a.fillna(-1) != _b.fillna(-1)).sum())} of {len(_series)} | "
                      f"NaN Fat_Suppression: {int(_a.isna().sum())}")
            _cfgs = {c.name: contract_config(c, _fs_used) for c in S7["caches"]}
            for _name, _cfg in _cfgs.items():
                S7["cache_dirs"][_name].mkdir(parents=True, exist_ok=True)
                print(f"  {_name}: layout {_cfg.layout}, {_cfg.n_slices} slices over {_cfg.band}, "
                      f"{_cfg.img} px, fs_negation {_cfg.fs_negation}, "
                      f"instance_fallback {_cfg.instance_fallback}")
            _todo = [s for s in sorted(_rows)
                     if not all((S7["cache_dirs"][n] / f"{s}.npz").is_file() for n in _cfgs)]
            _worker = JointWorker(_rows, str(_root / "test_series"),
                                  {n: str(d) for n, d in S7["cache_dirs"].items()}, _cfgs)
            _counts = {n: {} for n in _cfgs}
            _secs = {n: 0.0 for n in _cfgs}
            _t0 = time.time()
            print(f"{len(_rows)} studies | {len(_todo)} to build")
            # The pool pickles the worker and its configs by reference to __main__. A Mac
            # dry-run harness runs these cells in a namespace other than __main__'s, so the
            # two classes are registered there too; on Kaggle this is a no-op.
            _main = sys.modules["__main__"]
            if _main.__dict__ is not globals():
                for _cls in (JointWorker, CacheConfig):
                    setattr(_main, _cls.__name__, _cls)
            if _todo:
                with mp.get_context("fork").Pool(os.cpu_count() or 2) as _pool:
                    for _i, (_s, _recs) in enumerate(_pool.imap_unordered(_worker, _todo, chunksize=1), 1):
                        for _n, _rec in _recs.items():
                            _counts[_n][_rec["status"]] = _counts[_n].get(_rec["status"], 0) + 1
                            _secs[_n] += _rec["seconds"]
                            if _rec["status"] not in ("ok", "skip"):
                                print(f"  {_n} {_rec['status']}: {_s} {str(_rec['note'])[:200]}")
                        _el = time.time() - _t0
                        if _i in (25, 100) or _i % 250 == 0:
                            print(f"[{_el / 60:6.1f} min] {_i}/{len(_todo)} | {_el / _i:.2f} s/study | "
                                  f"projected {_el / _i * len(_todo) / 3600:.2f} h | {_counts}", flush=True)
                        if _el > 4 * 3600:
                            print(f"cache budget of 4 h reached at {_i}/{len(_todo)}")
                            _pool.terminate()
                            break
            _wall = time.time() - _t0
            S7["n_studies"]["cache"] = len(_todo)
            S7["build"] = {"wall": _wall, "seconds": _secs, "counts": _counts}
            print(f"caches built in {_wall / 60:.1f} min wall ({_wall / max(len(_todo), 1):.2f} s/study)")
            for _n in _cfgs:
                print(f"  {_n}: {_counts[_n]} | worker time {_secs[_n]:.1f} s "
                      f"(the first cache of a study pays the shared DICOM reads)")
        for _name, _dir in list(S7["cache_dirs"].items()):
            _n_files = len(list(Path(_dir).glob("*.npz")))
            print(f"cache {_name}: {_n_files} files in {_dir}")
            if _n_files == 0:
                _gone = [m for m in S7["members"] if m.contract.cache.name == _name]
                S7["members"] = [m for m in S7["members"] if m.contract.cache.name != _name]
                S7["dropped"].append(f"cache {_name} empty: dropped {len(_gone)} members")
                print(f"CACHE {_name} IS EMPTY: its {len(_gone)} members are dropped")
        if not S7["members"]:
            raise RuntimeError("no member has a usable cache")
    except Exception as exc:
        _s7_ours_failed(exc)
S7["times"]["cache"] = time.time() - _s7_t
print(f"cache stage: {S7['times']['cache'] / 60:.1f} min")

## Our stage, part 2: score the studies

The model classes are `knee/model.py`'s and the loaders and the blend are `knee/predict.py`'s, copied verbatim. A group's models are dealt across the GPUs round-robin and all launched before any result is read back. Each group's input is read once per study on a small thread pool that runs ahead of the GPUs; when the backbone side differs from the cache side, every triple is area-resized in [0, 1] before standardisation, exactly as `KneeCache._resize` did in training. Absent slots and series are skipped by the backbone (`SKIP_ABSENT`), which changes compute, not logits.

Budget: each group is warmed up on one study and timed on the next `PROBE_STUDIES`; the probe's predictions are kept. The projection is the time spent so far plus every group's measured cost over the remaining studies, against `BUDGET_H`. Over budget: one TTA pass, then whole sets from the bottom of `PRIORITY`, then folds; each drop is printed.

In [ ]:
import gc
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from scipy.stats import rankdata

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Every GPU the kernel has; each group's models are dealt across them round-robin.
DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print("torch", torch.__version__, "| timm", timm.__version__, "| devices", DEVICES)

# S6 hygiene (c): fp16 autocast, stated once; torch.cuda.is_bf16_supported() is never asked.
AMP_DTYPE = torch.float16
assert AMP_DTYPE is torch.float16
if torch.cuda.is_available():
    print(f"gpus: {torch.cuda.device_count()} x {torch.cuda.get_device_name(0)} | "
          f"capability {torch.cuda.get_device_capability(0)} | autocast dtype {AMP_DTYPE}")
else:
    print(f"no GPU: fp32 on the CPU (autocast dtype on GPU would be {AMP_DTYPE})")

gc.collect()
for _d in DEVICES:
    if _d.startswith("cuda"):
        with torch.cuda.device(_d):
            torch.cuda.empty_cache()
        print(f"  {_d}: {torch.cuda.memory_allocated(_d) / 2**30:.2f} GiB allocated, "
              f"{torch.cuda.memory_reserved(_d) / 2**30:.2f} GiB reserved")

In [ ]:
# --- verbatim from src/knee/model.py: N_META_V5, AttnPool, LabelAttnHead, AllSliceHead, SeriesBagHead, Model2p5D
N_META_V5 = 10

class AttnPool(nn.Module):
    """Gated attention over the slices of one slot (Ilse et al., attention MIL).

    One gate serves every slot and every label, so this pooling is label-agnostic and
    order-blind. See `LabelAttnHead` for the per-label alternative.

    Args:
        dim: Slice embedding width.
        hidden: Width of the gate MLP.
    """

    def __init__(self, dim, hidden):
        super().__init__()
        self.v = nn.Linear(dim, hidden)
        self.u = nn.Linear(dim, hidden)
        self.w = nn.Linear(hidden, 1)

    def forward(self, x):
        """Pools (B, S, D) slice embeddings into (B, D).

        Args:
            x: Slice embeddings of one slot per row.

        Returns:
            The pooled embedding and the (B, S) attention weights.
        """
        a = self.w(torch.tanh(self.v(x)) * torch.sigmoid(self.u(x)))   # (B, S, 1)
        a = a.softmax(dim=1)
        return (a * x).sum(dim=1), a.squeeze(-1)

class LabelAttnHead(nn.Module):
    """Per-label attention over every slice embedding in the study (attention-MIL).

    Each label owns one softmax over the K*S slice embeddings, so a finding visible on two
    slices of one slot is kept rather than averaged down by a gate shared across labels.
    Slices of absent slots are masked out of every softmax.

    Args:
        dim: Slice embedding width.
        hidden: Width of the attention MLP.
        n_slots: Number of cache slots K.
        n_out: Number of labels.
        slot_emb: Add a learned per-slot vector to each slice embedding before attention.
        dropout: Dropout inside the attention MLP.
    """

    def __init__(self, dim, hidden, n_slots, n_out, slot_emb, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.slot_emb = nn.Parameter(torch.zeros(n_slots, dim)) if slot_emb else None
        self.att = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Dropout(dropout),
                                 nn.Linear(hidden, n_out))
        self.cls_w = nn.Parameter(torch.empty(n_out, dim))
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.cls_w, std=0.02)

    def forward(self, f, present):
        """Pools (B, K, S, D) slice embeddings into (B, n_out) logits.

        Args:
            f: Slice embeddings, one per (slot, slice).
            present: (B, K) mask, 1 where the slot exists in the study.

        Returns:
            Logits of shape (B, n_out).
        """
        b, k, s, d = f.shape
        if self.slot_emb is not None:
            f = f + self.slot_emb[None, :, None, :]
        h = self.norm(f).reshape(b, k * s, d)
        a = self.att(h)                                                # (B, K*S, n_out)
        keep = (present > 0.5)[:, :, None].expand(b, k, s).reshape(b, k * s, 1)
        a = a.masked_fill(~keep, -1e4).softmax(dim=1)
        pooled = torch.einsum("bnl,bnd->bld", a, h)                    # (B, n_out, D)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

class AllSliceHead(nn.Module):
    """One gated attention over every slice of every slot in the study (I5 design a).

    The study is one bag of K*S slice embeddings rather than K pooled slots. A learned
    per-slot vector is added to each slice embedding so the gate knows which series type a
    slice came from, slices of absent slots are masked out of the softmax, and the pooled
    vector goes through dropout and one linear layer to the 12 logits. Label-agnostic, like
    `AttnPool`; see `LabelAttnHead` for the per-label variant.

    Args:
        dim: Slice embedding width.
        hidden: Width of the attention gate.
        n_slots: Number of cache slots K.
        n_out: Number of labels.
        dropout: Dropout on the pooled vector.
    """

    def __init__(self, dim, hidden, n_slots, n_out, dropout):
        super().__init__()
        self.slot_emb = nn.Parameter(torch.zeros(n_slots, dim))
        self.norm = nn.LayerNorm(dim)
        self.v = nn.Linear(dim, hidden)
        self.u = nn.Linear(dim, hidden)
        self.w = nn.Linear(hidden, 1)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(dim, n_out)

    def forward(self, f, present):
        """Pools (B, K, S, D) slice embeddings into (B, n_out) logits.

        Args:
            f: Slice embeddings, one per (slot, slice).
            present: (B, K) mask, 1 where the slot exists in the study.

        Returns:
            Logits of shape (B, n_out).
        """
        b, k, s, d = f.shape
        h = self.norm(f + self.slot_emb[None, :, None, :]).reshape(b, k * s, d)
        a = self.w(torch.tanh(self.v(h)) * torch.sigmoid(self.u(h)))     # (B, K*S, 1)
        keep = (present > 0.5)[:, :, None].expand(b, k, s).reshape(b, k * s, 1)
        a = a.masked_fill(~keep, -1e4).softmax(dim=1)
        pooled = (a * h).sum(dim=1)                                      # (B, D)
        return self.head(self.drop(pooled))

class SeriesBagHead(nn.Module):
    """Per-finding attention over every slice of every series in a cache-v5 bag.

    The study is one bag of N*S slice embeddings, N series of S slices each. Before
    attention each slice embedding gets a learned embedding of its series' type (plane,
    FS, contrast: the N_META_V5-wide one-hot from the loader) and, optionally, a learned
    embedding of its depth in the stack (binned over [0, 1]). Each of the n_out findings
    then owns one softmax over the bag, as in `LabelAttnHead`; slices of absent series are
    masked out of every softmax, so their pixels cannot reach the logits.

    Args:
        dim: Slice embedding width.
        hidden: Width of the attention MLP.
        n_meta: Width of the series-type vector.
        n_out: Number of labels.
        dropout: Dropout inside the attention MLP.
        meta_emb: Add the series-type embedding.
        depth_bins: Number of depth embeddings; 0 leaves depth out.
    """

    def __init__(self, dim, hidden, n_meta, n_out, dropout, meta_emb=True, depth_bins=16):
        super().__init__()
        self.meta = nn.Linear(n_meta, dim, bias=False) if meta_emb else None
        if self.meta is not None:
            nn.init.zeros_(self.meta.weight)
        self.depth = nn.Embedding(depth_bins, dim) if depth_bins else None
        if self.depth is not None:
            nn.init.zeros_(self.depth.weight)
        self.norm = nn.LayerNorm(dim)
        self.att = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Dropout(dropout),
                                 nn.Linear(hidden, n_out))
        self.cls_w = nn.Parameter(torch.empty(n_out, dim))
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.cls_w, std=0.02)

    def forward(self, f, present, meta=None, depth=None):
        """Pools (B, N, S, D) slice embeddings into (B, n_out) logits.

        Args:
            f: Slice embeddings, one per (series, slice).
            present: (B, N) mask, 1 where the bag position holds a series.
            meta: (B, N, n_meta) series types, or None to skip the type embedding.
            depth: (B, N, S) slice depths in [0, 1], or None to skip the depth embedding.

        Returns:
            Logits of shape (B, n_out).
        """
        b, n, s, d = f.shape
        if self.meta is not None and meta is not None:
            f = f + self.meta(meta.to(f.dtype))[:, :, None, :]
        if self.depth is not None and depth is not None:
            bins = (depth.float().clamp(0, 1) * (self.depth.num_embeddings - 1)).round().long()
            f = f + self.depth(bins).to(f.dtype)
        h = self.norm(f).reshape(b, n * s, d)
        a = self.att(h)                                                # (B, N*S, n_out)
        keep = (present > 0.5)[:, :, None].expand(b, n, s).reshape(b, n * s, 1)
        a = a.masked_fill(~keep, -1e4).softmax(dim=1)
        pooled = torch.einsum("bnl,bnd->bld", a, h)                    # (B, n_out, D)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

class Model2p5D(nn.Module):
    """2.5D study classifier: one encoder per slice, pooled per slot, one logit per label.

    Args:
        cfg: The training CFG dict; reads BACKBONE, PRETRAINED, VIT_*, GRAD_CHECKPOINT,
            N_SLOTS, HEAD (concat, label_attn, all_slice or series_bag), HEAD_HIDDEN,
            SLOT_EMB, ATTN_DIM, DROPOUT, DROP_PATH (optional), for series_bag META_EMB and
            DEPTH_BINS (optional), and for POS_EMB also CACHE_SLICES and SLICE_STRIDE.

    Raises:
        ValueError: POS_EMB with HEAD series_bag (the bag's depth embedding replaces it).
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_slots = cfg["N_SLOTS"]
        self.is_vit = "vit" in cfg["BACKBONE"]
        self.skip_absent = bool(cfg.get("SKIP_ABSENT", True))
        kw = {}
        if cfg.get("DROP_PATH"):
            kw["drop_path_rate"] = cfg["DROP_PATH"]
        if self.is_vit:
            kw["img_size"] = cfg["VIT_IMG"]
        elif "coatnet" in cfg["BACKBONE"]:
            # CoAtNet fixes its attention window at build time from img_size, so it must be
            # told the side it will be fed: MODEL_IMG when the dataset upsamples, else the
            # cache side.
            kw["img_size"] = cfg.get("MODEL_IMG") or cfg["IMG"]
        self.backbone = timm.create_model(
            cfg["BACKBONE"], pretrained=cfg["PRETRAINED"], num_classes=0, in_chans=3, **kw)
        if cfg["GRAD_CHECKPOINT"]:
            self.backbone.set_grad_checkpointing(True)
        dim = self.backbone.num_features
        if self.is_vit:
            # DINOv2 recipe: freeze everything but the last VIT_UNFREEZE blocks (+ final
            # norm); feature = [CLS ; mean of patch tokens]. VIT_UNFREEZE >= depth means a
            # genuinely full fine-tune, with patch_embed, cls_token and pos_embed trainable
            # too, so that a ViT can be compared fairly against a fully trained convnext.
            n_blocks = len(self.backbone.blocks)
            if cfg["VIT_UNFREEZE"] < n_blocks:
                for p in self.backbone.parameters():
                    p.requires_grad = False
                for blk in self.backbone.blocks[-cfg["VIT_UNFREEZE"]:]:
                    for p in blk.parameters():
                        p.requires_grad = True
                for p in self.backbone.norm.parameters():
                    p.requires_grad = True
            else:
                print(f"ViT: FULL fine-tune ({n_blocks} blocks, nothing frozen)")
            dim = dim * 2
        # A learned vector per (slot, sampled slice position), added before pooling. The pools
        # are order-blind and the right-knee mirror reverses sagittal slice order, so without
        # this the model cannot tell where in the stack a slice sits.
        self.pos_emb = None
        if cfg.get("POS_EMB") and cfg["HEAD"] == "series_bag":
            raise ValueError("POS_EMB is a slot-cache option; series_bag uses DEPTH_BINS")
        if cfg.get("POS_EMB"):
            n_pos = len(range(0, cfg["CACHE_SLICES"], cfg["SLICE_STRIDE"]))
            self.pos_emb = nn.Parameter(torch.zeros(self.n_slots, n_pos, dim))
        self.head_kind = cfg["HEAD"]
        if self.head_kind == "label_attn":
            self.head = LabelAttnHead(dim, cfg["HEAD_HIDDEN"], self.n_slots, len(TARGETS),
                                      cfg["SLOT_EMB"], cfg["DROPOUT"])
        elif self.head_kind == "all_slice":
            self.head = AllSliceHead(dim, cfg["HEAD_HIDDEN"], self.n_slots, len(TARGETS),
                                     cfg["DROPOUT"])
        elif self.head_kind == "series_bag":
            self.head = SeriesBagHead(dim, cfg["HEAD_HIDDEN"], N_META_V5, len(TARGETS),
                                      cfg["DROPOUT"], cfg.get("META_EMB", True),
                                      cfg.get("DEPTH_BINS", 16))
        else:
            self.pool = AttnPool(dim, cfg["ATTN_DIM"])
            self.drop = nn.Dropout(cfg["DROPOUT"])
            self.head = nn.Linear(dim * self.n_slots, len(TARGETS))

    def forward(self, x, present=None, meta=None, depth=None):
        """Scores a batch of studies.

        Args:
            x: uint8 or float images of shape (B, slots, S, 3, H, W); for series_bag the
                slot axis is the bag of series.
            present: (B, slots) mask, 1 where the slot exists in the study.
            meta: series_bag only: (B, slots, N_META_V5) series types; ignored otherwise.
            depth: series_bag only: (B, slots, S) slice depths in [0, 1]; ignored otherwise.

        Returns:
            Logits of shape (B, 12).
        """
        b, k, s = x.shape[:3]
        flat = x.reshape(b * k * s, *x.shape[3:])
        if self.is_vit:
            t = self.backbone.forward_features(flat)                   # (N, 1+P, D)
            npre = self.backbone.num_prefix_tokens
            f = torch.cat([t[:, 0], t[:, npre:].mean(dim=1)], dim=1)   # (N, 2D)
        elif present is not None and self.skip_absent:
            # Run the backbone only on present slots. An absent slot is all zeros in the
            # cache and its embedding is zeroed or masked below anyway, so this changes
            # nothing but the compute (about a fifth of the images in training).
            keep = present.reshape(b * k) > 0.5
            per_slot = x.reshape(b * k, s, *x.shape[3:])[keep]         # (M, S, 3, H, W)
            if per_slot.shape[0]:
                fk = self.backbone(per_slot.reshape(-1, *x.shape[3:]))  # (M*S, D)
                f = fk.new_zeros((b * k, s, fk.shape[-1]))
                f[keep] = fk.reshape(-1, s, fk.shape[-1])
            else:
                f = x.new_zeros((b * k, s, self.backbone.num_features))
        else:
            f = self.backbone(flat)                                    # (B*k*S, D)
        f = f.reshape(b, k, s, -1)
        if self.pos_emb is not None:
            f = f + self.pos_emb[None, :, :s]
        if self.head_kind == "series_bag":
            if present is None:
                present = torch.ones(b, k, device=f.device)
            return self.head(f, present, meta, depth)
        if self.head_kind in ("label_attn", "all_slice"):
            if present is None:
                present = torch.ones(b, k, device=f.device)
            return self.head(f, present)
        f = f.reshape(b * k, s, -1)
        emb, _ = self.pool(f)                                          # (B*k, D)
        emb = emb.reshape(b, k, -1)
        if present is not None:
            # A missing slot is all zeros in the cache; zero its embedding too, so the head
            # sees "absent" rather than "the embedding of a black volume".
            emb = emb * present.unsqueeze(-1)
        return self.head(self.drop(emb.reshape(b, -1)))

In [ ]:
# The v5 bag reader (knee/dataset.py) and the two helpers it needs (knee/cache.py), copied
# again here because the public stage, when on, runs between the cache cells and this one.
# --- verbatim from src/knee/cache.py: PLANES_V5, CONTRASTS_V5, TYPE_ORDER_V5, select_series, series_meta_vector
PLANES_V5 = ("Sagittal", "Coronal", "Axial")

CONTRASTS_V5 = ("T1", "PD", "T2", "STIR", "GRE", "UNK")

TYPE_ORDER_V5 = ((0, 1), (1, 1), (2, 1), (0, 0), (1, 0), (2, 0))

def select_series(plane, fs, contrast, slot, n_max, typed=False):
    """Chooses which of a study's stored series fill a bag of n_max, in bag order.

    Pass 1 takes one series per (plane, FS) type in TYPE_ORDER_V5 order, preferring the one
    holding the lowest v3 slot, then the earliest stored. Pass 2 adds every other series
    that holds a v3 slot, in slot order, so a bag of six or more always contains what the
    six-slot caches saw. Pass 3 takes series whose (plane, FS, contrast) is not yet in the
    bag, in stored order. Pass 4 fills what is left in stored order. Deterministic: the same
    study always gives the same bag.

    Packed (typed False), the bag is the passes' series in that order, so positions 0-2
    hold the sagittal, coronal and axial FS series whenever all three exist. Typed, the
    first six positions are reserved for the six pass-1 types (-1 where the study has none)
    and positions 6 onward take passes 2-4, so a position means the same series type in
    every study, as the six-slot caches' positions do.

    Args:
        plane: (N,) plane indices into PLANES_V5 (-1 unknown).
        fs: (N,) FS flags.
        contrast: (N,) contrast indices into CONTRASTS_V5.
        slot: (N,) v3 slot ids, -1 for none.
        n_max: Bag size; at least 6 when typed.
        typed: Reserve one position per (plane, FS) type.

    Returns:
        Indices into the stored series in bag order: up to n_max of them when packed,
        exactly n_max with -1 for an empty position when typed.

    Raises:
        ValueError: typed with n_max below the number of types.
    """
    n = len(plane)
    first = []
    for p, f in TYPE_ORDER_V5:
        cands = [i for i in range(n) if int(plane[i]) == p and int(fs[i]) == f]
        cands.sort(key=lambda i: (int(slot[i]) if int(slot[i]) >= 0 else 99, i))
        first.append(cands[0] if cands else -1)
    taken = {i for i in first if i >= 0}
    rest = []
    for i in sorted((i for i in range(n) if int(slot[i]) >= 0), key=lambda i: int(slot[i])):
        if i not in taken:
            rest.append(i)
            taken.add(i)
    seen = {(int(plane[i]), int(fs[i]), int(contrast[i])) for i in taken}
    for i in range(n):
        key = (int(plane[i]), int(fs[i]), int(contrast[i]))
        if i not in taken and key not in seen:
            rest.append(i)
            taken.add(i)
            seen.add(key)
    rest += [i for i in range(n) if i not in taken]
    if typed:
        if n_max < len(TYPE_ORDER_V5):
            raise ValueError(f"a typed bag needs n_max >= {len(TYPE_ORDER_V5)}, got {n_max}")
        return first + (rest + [-1] * n_max)[: n_max - len(first)]
    return ([i for i in first if i >= 0] + rest)[:n_max]

def series_meta_vector(plane, fs, contrast):
    """Encodes series types as the model's metadata input.

    Args:
        plane: (N,) plane indices into PLANES_V5 (-1 unknown gives an all-zero one-hot).
        fs: (N,) FS flags.
        contrast: (N,) contrast indices into CONTRASTS_V5.

    Returns:
        A float32 (N, N_META_V5) array: plane one-hot, FS, contrast one-hot.
    """
    n = len(plane)
    out = np.zeros((n, N_META_V5), np.float32)
    for i in range(n):
        if 0 <= int(plane[i]) < len(PLANES_V5):
            out[i, int(plane[i])] = 1.0
        out[i, len(PLANES_V5)] = float(fs[i])
        out[i, len(PLANES_V5) + 1 + int(contrast[i])] = 1.0
    return out
# --- verbatim from src/knee/dataset.py: load_series_bag
def load_series_bag(path, n_max: int, typed: bool = True):
    """Reads one v5 cache file into a fixed-size bag of series.

    Only the chosen series are decompressed. The function is pure numpy so that the
    submission kernel can carry a verbatim copy.

    Args:
        path: The study's v5 .npz.
        n_max: Bag size.
        typed: One reserved position per (plane, FS) type (see knee.cache.select_series).

    Returns:
        vol, a (n_max, n_slices, H, W) uint8 array, zeros at empty positions; present, a
        (n_max,) float32 mask; meta, a (n_max, N_META_V5) float32 array of series types
        (zeros at empty positions); and slice_pos, a (n_max, n_slices) float32 array of
        each stored slice's fraction of its stack.
    """
    with np.load(path) as d:
        bag = select_series(d["plane"], d["fs"], d["contrast"], d["slot"], n_max, typed)
        first = d[f"vol_{next(i for i in bag if i >= 0):02d}"]
        vol = np.zeros((n_max, *first.shape), np.uint8)
        present = np.zeros(n_max, np.float32)
        types = series_meta_vector(d["plane"], d["fs"], d["contrast"])
        meta = np.zeros((n_max, types.shape[1]), np.float32)
        pos = np.zeros((n_max, first.shape[0]), np.float32)
        slice_pos = d["slice_pos"].astype(np.float32)
        for k, i in enumerate(bag):
            if i < 0:
                continue
            vol[k] = d[f"vol_{i:02d}"]
            present[k] = 1.0
            meta[k] = types[i]
            pos[k] = slice_pos[i]
    return vol, present, meta, pos

In [ ]:
# --- verbatim from src/knee/predict.py: MODEL_DEFAULTS, INFER_OVERRIDES, PIXEL_MEAN, model_from_checkpoint, resize_views, _triples, load_slot_views, load_series_views, load_views, load_group_models, predict_group, member_ranks, _pct_rank, family_blend
MODEL_DEFAULTS = {
    "N_SLOTS": 6, "ATTN_DIM": 128, "DROPOUT": 0.2, "HEAD": "concat", "HEAD_HIDDEN": 256,
    "SLOT_EMB": False, "POS_EMB": False, "SLICE_STRIDE": 2, "IMG": 320, "MODEL_IMG": None,
    "CACHE_SLICES": 24, "CACHE_KEY": "rsna-knee-cache-v3", "SLOTS": None, "VIT_IMG": 224,
    "CACHE_FORMAT": "slots", "N_SERIES_MAX": 8, "SERIES_LAYOUT": "typed", "META_EMB": True,
    "DEPTH_BINS": 16,
}

INFER_OVERRIDES = {"PRETRAINED": False, "GRAD_CHECKPOINT": False}

PIXEL_MEAN, PIXEL_STD = 0.449, 0.226

def model_from_checkpoint(path, device: str):
    """Builds a model from the training config stored in a checkpoint and loads its weights.

    Args:
        path: A checkpoint written by `knee.train` (holds `model` and `cfg`).
        device: Torch device string.

    Returns:
        The model in eval mode and the config it was built from.

    Raises:
        SystemExit: The checkpoint carries no training config.
    """
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if "cfg" not in ckpt or "model" not in ckpt:
        raise SystemExit(f"{path} carries no training config; cannot rebuild its architecture")
    cfg = dict(MODEL_DEFAULTS)
    cfg.update({k: v for k, v in ckpt["cfg"].items() if v is not None})
    cfg.update(INFER_OVERRIDES)
    model = Model2p5D(cfg).to(device).eval()
    model.load_state_dict(ckpt["model"], strict=True)
    return model, cfg

def resize_views(x: np.ndarray, side: int, present=None) -> np.ndarray:
    """Resizes every image of a batch of views with area interpolation, as the trainer does.

    The trainer (`KneeCache._resize`) resizes the [0, 1] triples before standardising them;
    this does the same. Absent slots are all zeros before and after, so they are skipped.

    Args:
        x: Float32 triples in [0, 1] of shape (V, K, S, 3, H, W).
        side: Output side.
        present: (K,) presence flags, or None to resize every slot.

    Returns:
        A float32 array of shape (V, K, S, 3, side, side).
    """
    import cv2

    v, k, s, c, h, w = x.shape
    if h == side and w == side:
        return x
    out = np.zeros((v, k, s, c, side, side), np.float32)
    slots = range(k) if present is None else [i for i in range(k) if present[i] > 0.5]
    for i in slots:
        flat = x[:, i].reshape(-1, h, w)
        out[:, i] = np.stack([cv2.resize(f, (side, side), interpolation=cv2.INTER_AREA)
                              for f in flat]).reshape(v, s, c, side, side)
    return out

def _triples(vol: np.ndarray, idx: np.ndarray) -> np.ndarray:
    """Builds the 2.5D triple (i-1, i, i+1) of every sampled slice, scaled to [0, 1].

    Args:
        vol: uint8 (K, n, H, W).
        idx: Sample indices, (S,) shared by every slot or series.

    Returns:
        Float32 (K, S, 3, H, W).
    """
    n = vol.shape[1]
    x = np.stack([vol[:, np.clip(idx - 1, 0, n - 1)], vol[:, idx],
                  vol[:, np.clip(idx + 1, 0, n - 1)]], axis=2)
    return x.astype(np.float32) / 255.0

def load_slot_views(path, offsets, contract: InputContract):
    """Reads one slot-cache study as a batch of TTA views for one contract.

    Same as `load_study` without the slice-order reversal (submission 6 hygiene a), plus the
    slot subset and the resize to the backbone side.

    Args:
        path: The study's slot-cache npz.
        offsets: Slice-sampling offsets, one view each.
        contract: The group's input contract.

    Returns:
        A dict with 'x' (V, K, S, 3, side, side) float32 standardised and 'present' (K,)
        float32.
    """
    with np.load(path) as d:
        vol, present = d["vol"], d["present"].astype(np.float32)
    if contract.slots is not None:
        sel = np.asarray(contract.slots, dtype=int)
        vol, present = vol[sel], present[sel]
    n = vol.shape[1]
    base = np.arange(0, n, contract.stride)
    x = np.stack([_triples(vol, np.clip(base + off, 0, n - 1)) for off in offsets])
    x = resize_views(x, contract.side, present)
    return {"x": (x - PIXEL_MEAN) / PIXEL_STD, "present": present}

def load_series_views(path, offsets, contract: InputContract):
    """Reads one v5 study as a batch of TTA views of its series bag.

    The bag is `knee.dataset.load_series_bag`'s; each view samples the stride grid shifted by
    its offset in every series, and carries the sampled slices' depths, as `KneeSeriesBag`
    does at validation (offset 0).

    Args:
        path: The study's v5 npz.
        offsets: Slice-sampling offsets, one view each.
        contract: The group's input contract.

    Returns:
        A dict with 'x' (V, N, S, 3, side, side) float32 standardised, 'present' (N,),
        'meta' (N, 10) and 'depth' (V, N, S), all float32.
    """
    vol, present, meta, pos = load_series_bag(path, contract.n_series, contract.typed)
    n = vol.shape[1]
    base = np.arange(0, n, contract.stride)
    xs, depths = [], []
    for off in offsets:
        idx = np.clip(base + off, 0, n - 1)
        xs.append(_triples(vol, idx))
        depths.append(pos[:, idx])
    x = resize_views(np.stack(xs), contract.side, present)
    return {"x": (x - PIXEL_MEAN) / PIXEL_STD, "present": present, "meta": meta,
            "depth": np.stack(depths).astype(np.float32)}

def load_views(cache_dir, study: str, offsets, contract: InputContract):
    """Reads one study for one contract from its cache directory.

    Args:
        cache_dir: The contract's cache directory.
        study: StudyInstanceUID.
        offsets: Slice-sampling offsets.
        contract: The group's input contract.

    Returns:
        The loader's dict, or None when the study has no cache file.
    """
    path = Path(cache_dir) / f"{study}.npz"
    if not path.is_file():
        return None
    if contract.cache.layout == "series":
        return load_series_views(path, offsets, contract)
    return load_slot_views(path, offsets, contract)

def load_group_models(members, devices):
    """Builds and loads the models of one group, dealt across the devices round-robin.

    Args:
        members: The group's members.
        devices: Torch device strings.

    Returns:
        The models in eval mode and the members they belong to, in member order. A
        checkpoint that fails to build or load strictly is skipped with a printed line, so
        it costs itself, not its group.
    """
    models, kept = [], []
    for m in members:
        try:
            model, _ = model_from_checkpoint(m.path, devices[len(models) % len(devices)])
        # Deliberate isolation point: one unloadable checkpoint must not cost the group.
        except Exception as exc:
            print(f"SKIPPED {m.set_name}/{m.path.name}: cannot load: {type(exc).__name__}: "
                  f"{str(exc)[:300]}")
            continue
        models.append(model)
        kept.append(m)
    return models, kept

@torch.no_grad()
def predict_group(models, studies, offsets, contract: InputContract, cache_dir,
                  amp_dtype=torch.float16, views_per_call: int | None = None,
                  prefetch: int = 4) -> np.ndarray:
    """Runs one group's models over the studies, averaging each over its TTA views.

    Each study is read once for the whole group; the reads run ahead on a thread pool while
    the GPUs work. Every model is launched before any result is read back, so models on
    different GPUs run at the same time.

    Args:
        models: The group's loaded models (each on its own device).
        studies: StudyInstanceUIDs.
        offsets: Slice-sampling offsets.
        contract: The group's input contract.
        cache_dir: The contract's cache directory.
        amp_dtype: Autocast dtype on CUDA.
        views_per_call: Split the TTA views into forward calls of at most this many (eval
            mode has no batch coupling, so this changes memory, not results); None runs all
            views at once.
        prefetch: Studies read ahead.

    Returns:
        Probabilities (len(models), len(studies), 12); NaN rows where a study has no cache
        file or its file cannot be read (the first such error is printed).
    """
    out = np.full((len(models), len(studies), len(TARGETS)), np.nan, np.float32)
    devices = sorted({str(next(m.parameters()).device) for m in models})
    step = views_per_call or len(offsets)
    n_bad, t0 = 0, time.time()
    with ThreadPoolExecutor(max_workers=max(1, min(prefetch, 4))) as pool:
        futures = [pool.submit(load_views, cache_dir, s, offsets, contract)
                   for s in studies[:prefetch]]
        for i in range(len(studies)):
            try:
                loaded = futures[i].result()
            # A study whose cache file cannot be read costs that study (its row stays NaN),
            # not the group.
            except Exception as exc:
                loaded = None
                n_bad += 1
                if n_bad == 1:
                    print(f"  {contract.name}: cannot read {studies[i]}: "
                          f"{type(exc).__name__}: {exc}")
            futures[i] = None
            if i + prefetch < len(studies):
                futures.append(pool.submit(load_views, cache_dir, studies[i + prefetch],
                                           offsets, contract))
            if loaded is None:
                continue
            probs = [[] for _ in models]
            for v0 in range(0, len(offsets), step):
                v1 = min(v0 + step, len(offsets))
                feed = {}
                for d in devices:
                    kw = {"present": torch.from_numpy(loaded["present"]).to(d)
                          .unsqueeze(0).expand(v1 - v0, -1)}
                    if "meta" in loaded:
                        kw["meta"] = (torch.from_numpy(loaded["meta"]).to(d)
                                      .unsqueeze(0).expand(v1 - v0, -1, -1))
                        kw["depth"] = torch.from_numpy(loaded["depth"][v0:v1]).to(d)
                    xb = torch.from_numpy(np.ascontiguousarray(loaded["x"][v0:v1])).to(d)
                    feed[d] = (xb, kw)
                for j, model in enumerate(models):
                    d = str(next(model.parameters()).device)
                    xb, kw = feed[d]
                    with torch.autocast("cuda", dtype=amp_dtype, enabled=d.startswith("cuda")):
                        logits = model(xb, **kw)
                    probs[j].append(torch.sigmoid(logits.float()))
            for j, p in enumerate(probs):
                out[j, i] = torch.cat(p).mean(dim=0).cpu().numpy()
            if (i + 1) % 200 == 0:
                rate = (time.time() - t0) / (i + 1)
                print(f"  {contract.name}: {i + 1}/{len(studies)} | {rate:.2f} s/study | "
                      f"eta {rate * (len(studies) - i - 1) / 60:.1f} min", flush=True)
    return out

def member_ranks(preds: np.ndarray) -> np.ndarray:
    """Turns each member's probabilities into per-label percentile ranks.

    Each member is ranked over the studies it scored; a study it could not score stays NaN.

    Args:
        preds: (members, studies, 12) probabilities, NaN rows allowed.

    Returns:
        Ranks in (0, 1], same shape, NaN where the input row was NaN.
    """
    out = np.full(preds.shape, np.nan, np.float64)
    for m in range(preds.shape[0]):
        ok = np.isfinite(preds[m]).all(axis=1)
        if ok.any():
            out[m, ok] = rankdata(preds[m, ok], axis=0) / ok.sum()
    return out

def _pct_rank(scores: np.ndarray) -> np.ndarray:
    """Per-label percentile ranks over the finite rows, ties averaged.

    Args:
        scores: (studies, 12), NaN rows allowed.

    Returns:
        Ranks in (0, 1], NaN where the row was NaN.
    """
    out = np.full(scores.shape, np.nan, np.float64)
    ok = np.isfinite(scores).all(axis=1)
    if ok.any():
        out[ok] = rankdata(scores[ok], axis=0) / ok.sum()
    return out

def family_blend(preds: np.ndarray, families, weights: dict | None = None):
    """Rank-averages members within each family, then families with fixed weights.

    Within a family the members' percentile ranks are averaged (a study one member could not
    score is averaged over the members that did); each family's mean is ranked again, and the
    families' ranks are averaged with `weights` (1 for a family not listed) over the families
    that scored the study. A study no family scored gets 0.5.

    Args:
        preds: (members, studies, 12) probabilities.
        families: The family of each member, in member order.
        weights: Family -> weight, or None for equal weights.

    Returns:
        The (studies, 12) blend in (0, 1], and a dict family -> its (studies, 12) ranks.
    """
    ranks = member_ranks(preds)
    names = list(dict.fromkeys(families))
    per_family = {}
    for f in names:
        idx = [i for i, g in enumerate(families) if g == f]
        with np.errstate(all="ignore"):
            mean = np.nanmean(ranks[idx], axis=0)
        per_family[f] = _pct_rank(mean)
    w = np.array([(weights or {}).get(f, 1.0) for f in names], np.float64)
    stack = np.stack([per_family[f] for f in names])                    # (F, studies, 12)
    have = np.isfinite(stack)
    num = (np.where(have, stack, 0.0) * w[:, None, None]).sum(axis=0)
    den = (have * w[:, None, None]).sum(axis=0)
    final = np.full(den.shape, 0.5)
    np.divide(num, den, out=final, where=den > 0)
    return final, per_family

In [ ]:
def study_list():
    """The studies to score, in sample_submission order.

    Returns:
        StudyInstanceUIDs (the first `studies` of them in a dry run that caps them).
    """
    sub = pd.read_csv(_s7_comp_root() / "sample_submission.csv")
    ids = sub["StudyInstanceUID"].astype(str).tolist()
    if S7["dry_run"] and S7["dry_run_studies"]:
        ids = ids[:S7["dry_run_studies"]]
    return ids


def audit_cache(contract_name, studies):
    """Counts absent slots or series and missing cache files over the studies to score.

    S6 hygiene (d), per cache.

    Args:
        contract_name: The cache name.
        studies: StudyInstanceUIDs.

    Returns:
        A one-line summary.
    """
    d = Path(S7["cache_dirs"][contract_name])
    missing, absent, n_series = 0, None, []
    for s in studies:
        p = d / f"{s}.npz"
        if not p.is_file():
            missing += 1
            continue
        with np.load(p) as z:
            if "present" in z.files:
                a = (np.asarray(z["present"]).reshape(-1) == 0).astype(int)
                absent = a if absent is None else absent + a
            else:
                n_series.append(int(z["n_series"]))
    if n_series:
        return (f"{contract_name}: series per study mean {np.mean(n_series):.2f} "
                f"(min {min(n_series)}, max {max(n_series)}) | no cache file: {missing}")
    return (f"{contract_name}: absent per slot {absent.tolist() if absent is not None else []}"
            f" | no cache file: {missing}")


def run_group(models, studies, offsets, contract):
    """Scores studies with one group, splitting the TTA views for large inputs.

    Args:
        models: The group's loaded models.
        studies: StudyInstanceUIDs.
        offsets: TTA offsets.
        contract: The group's input contract.

    Returns:
        (members, studies, 12) probabilities.
    """
    per_call = 1 if contract.side >= ONE_VIEW_MIN_SIDE else None
    return predict_group(models, studies, offsets, contract,
                         S7["cache_dirs"][contract.cache.name], AMP_DTYPE, per_call)

In [ ]:
# Score every study with every group.
_s7_t = time.time()
if S7["ours_active"]:
    try:
        studies = study_list()
        S7["n_studies"]["ours"] = len(studies)
        print(f"{len(studies)} studies to predict")
        for _name in S7["cache_dirs"]:
            print("  " + audit_cache(_name, studies))

        offsets = list(S7["tta_offsets"])
        public_h = S7["times"].get("public", 0.0) / 3600
        if PUBLIC_STAGE and public_h > S7["public_tta2_max_h"] and len(offsets) > 1:
            offsets = offsets[:1]
            print(f"public stage took {public_h:.2f} h > {S7['public_tta2_max_h']} h: one TTA pass")

        # Load each group, warm it up on one study, time it on the next PROBE_STUDIES.
        GROUPS, done, cost = {}, {}, {}
        for contract, ms in group_members(S7["members"]).items():
            try:
                t0 = time.time()
                models, loaded = load_group_models(ms, DEVICES)
                for m in ms:
                    if m not in loaded:
                        S7["dropped"].append(f"{m.set_name}/{m.path.name}: failed to load")
                S7["members"] = [m for m in S7["members"] if m.contract != contract or m in loaded]
                ms = loaded
                if not models:
                    raise RuntimeError("no checkpoint of the group loads")
                t_load = time.time() - t0
                first = studies[:1]
                probe = studies[1:1 + PROBE_STUDIES]
                p_first = run_group(models, first, offsets, contract)
                t0 = time.time()
                p_probe = run_group(models, probe, offsets, contract)
                per_study = (time.time() - t0) / max(len(probe), 1)
                hints = np.array([pass_hint(m) for m in ms])
                for m, h in zip(ms, hints):
                    cost[m.path] = per_study / len(offsets) * h / hints.sum()
                GROUPS[contract] = {"members": ms, "models": models, "probed": first + probe}
                done[contract] = np.concatenate([p_first, p_probe], axis=1)
                print(f"group {contract.name}: {len(ms)} models loaded in {t_load:.0f} s | "
                      f"probe {per_study:.2f} s/study at {len(offsets)} TTA | "
                      f"{per_study / len(ms) / len(offsets):.3f} s per member-pass | "
                      f"input {tuple(load_views(S7['cache_dirs'][contract.cache.name], probe[0] if probe else first[0], offsets[:1], contract)['x'].shape)}")
            except Exception as exc:
                print("=" * 78)
                print(f"S7 GROUP FAILED {contract.name}: {type(exc).__name__}: {exc}")
                traceback.print_exception(type(exc), exc, exc.__traceback__, limit=4)
                print("=" * 78)
                S7["dropped"].append(f"group {contract.name} failed: {type(exc).__name__}: {exc}")
                S7["members"] = [m for m in S7["members"] if m.contract != contract]
        if not GROUPS:
            raise RuntimeError("every group failed to load or probe")

        n_rest = max(len(studies) - 1 - PROBE_STUDIES, 0)
        left = BUDGET_H * 3600 - (time.time() - NB_T0)
        members = [m for g in GROUPS.values() for m in g["members"]]
        proj = projected_seconds(members, cost, n_rest, len(offsets))
        print(f"projection: {proj / 3600:.2f} h for the remaining {n_rest} studies at "
              f"{len(offsets)} TTA | {left / 3600:.2f} h left of {BUDGET_H} h")
        kept, n_off, log = trim_members(members, cost, n_rest, left, len(offsets), PRIORITY)
        for line in log:
            print(f"OVER BUDGET: {line}")
            S7["dropped"].append(f"probe: {line}")
        if n_off != len(offsets):
            offsets = offsets[:n_off]
            done = {}                      # the probe used more views: score those studies again
        kept_paths = {m.path for m in kept}
        S7["run"] = {"members": len(kept), "offsets": list(offsets),
                     "projected_h": projected_seconds(kept, cost, n_rest, n_off) / 3600}
        print(f"running {len(kept)} members x {len(offsets)} TTA on {len(DEVICES)} device(s) | "
              f"projected {S7['run']['projected_h']:.2f} h")

        preds, families, sets = [], [], []
        for contract, g in GROUPS.items():
            keep = [i for i, m in enumerate(g["members"]) if m.path in kept_paths]
            if not keep:
                print(f"group {contract.name}: every member dropped")
                continue
            models = [g["models"][i] for i in keep]
            ms = [g["members"][i] for i in keep]
            t0 = time.time()
            try:
                if contract in done:
                    rest = studies[len(g["probed"]):]
                    p_rest = run_group(models, rest, offsets, contract)
                    p = np.concatenate([done[contract][keep], p_rest], axis=1)
                else:
                    p = run_group(models, studies, offsets, contract)
            except Exception as exc:
                print(f"S7 GROUP FAILED {contract.name} while scoring: {type(exc).__name__}: {exc}")
                traceback.print_exception(type(exc), exc, exc.__traceback__, limit=4)
                S7["dropped"].append(f"group {contract.name} failed while scoring: {exc}")
                continue
            ok = np.isfinite(p).all(axis=(0, 2))
            print(f"group {contract.name}: {len(ms)} members x {len(offsets)} TTA in "
                  f"{(time.time() - t0) / 60:.1f} min | {ok.sum()}/{len(studies)} studies scored")
            preds.append(p)
            families += [m.family for m in ms]
            sets += [m.set_name for m in ms]
            g["models"] = models = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        if not preds:
            raise RuntimeError("no group produced predictions")
        preds = np.concatenate(preds)
        final, per_family = family_blend(preds, families, FAMILY_WEIGHT)
        scored = np.isfinite(preds).all(axis=2).any(axis=0)
        print(f"families {list(per_family)} | weights "
              f"{[FAMILY_WEIGHT.get(f, 1.0) for f in per_family]} | "
              f"{int(scored.sum())}/{len(studies)} studies scored by at least one member; "
              f"{int((~scored).sum())} fall back to 0.5")
        if len(per_family) > 1:
            names = list(per_family)
            for i, a in enumerate(names):
                for b in names[i + 1:]:
                    both = np.isfinite(per_family[a]).all(1) & np.isfinite(per_family[b]).all(1)
                    rho = [pd.Series(per_family[a][both, j]).corr(pd.Series(per_family[b][both, j]),
                                                                  method="spearman")
                           for j in range(len(TARGETS))]
                    print(f"  rank correlation {a} vs {b}: mean {np.nanmean(rho):.3f}")
        ours = pd.DataFrame(final, columns=TARGETS)
        ours.insert(0, "StudyInstanceUID", studies)
        S7["ours_csv"] = S7["work_dir"] / "_s7_ours.csv"
        ours.to_csv(S7["ours_csv"], index=False)
        np.save(S7["work_dir"] / "_s7_member_preds.npy", preds)
        pd.DataFrame({"set": sets, "family": families}).to_csv(
            S7["work_dir"] / "_s7_members.csv", index=False)
        S7["ours_ok"] = True
        S7["n_members_run"] = len(families)
        print(f"our stage: {ours.shape} -> {S7['ours_csv']}")
        del GROUPS
        gc.collect()
    except (Exception, SystemExit) as exc:
        _s7_ours_failed(exc)
S7["times"]["ours"] = time.time() - _s7_t
print(f"our scoring stage: {S7['times']['ours'] / 60:.1f} min")

## Stage 3: rank blend, checks, timings

Per finding, each side is ranked (ties averaged, as a fraction) over its own non-failed studies only: ours over the studies at least one member scored, Raptor over the studies whose row is not 0.5 in all twelve columns. `final = W_OURS * rank(ours) + (1 - W_OURS) * rank(Raptor)` with `W_OURS = 0.5` for every finding. A study one side failed takes the other side's rank; a study both failed gets 0.5. If the Raptor stage failed, or its table fails validation, or it failed on every study, the submission is submission 11b's own write (`s7_rank` of our family blend) and the log says `BLEND_FALLBACK raptor`.

In [ ]:
def s7_rank(frame, ids, labels):
    """Per-label percentile ranks of a score table, aligned to a study order.

    Args:
        frame: A table with StudyInstanceUID and the label columns.
        ids: StudyInstanceUIDs in output order; every one must be in `frame`.
        labels: The label columns.

    Returns:
        A (len(ids), len(labels)) float array of ranks in (0, 1], ties averaged.

    Raises:
        ValueError: A study is missing, duplicated, or a value is not finite.
    """
    frame = frame.copy()
    frame["StudyInstanceUID"] = frame["StudyInstanceUID"].astype(str)
    if frame["StudyInstanceUID"].duplicated().any():
        raise ValueError("duplicate StudyInstanceUID")
    aligned = frame.set_index("StudyInstanceUID").reindex(list(ids))[list(labels)]
    if aligned.isna().any().any():
        raise ValueError(f"{int(aligned.isna().any(axis=1).sum())} studies missing or NaN")
    if not np.isfinite(aligned.to_numpy(np.float64)).all():
        raise ValueError("non-finite scores")
    return aligned.rank(method="average", pct=True).to_numpy(np.float64)


def s13_rank_on(values, ok):
    """Percentile ranks of one score column over a subset of its rows.

    Args:
        values: Scores, one per study.
        ok: Boolean mask of the rows to rank (the side's non-failed studies).

    Returns:
        A float array of len(values): ranks in (0, 1] on the `ok` rows (ties averaged),
        NaN elsewhere.
    """
    values = np.asarray(values, dtype=np.float64)
    ranks = np.full(len(values), np.nan)
    if ok.any():
        ranks[ok] = pd.Series(values[ok]).rank(method="average", pct=True).to_numpy()
    return ranks


def s13_blend(ours, ours_fail, raptor, raptor_fail, w_ours):
    """The stage 3 blend of two aligned score tables.

    Args:
        ours: (n, k) array, our family blend.
        ours_fail: (n,) bool, studies no member of ours scored.
        raptor: (n, k) array, Raptor's raw probabilities.
        raptor_fail: (n,) bool, studies Raptor failed on.
        w_ours: Weight of our rank; Raptor's is 1 - w_ours.

    Returns:
        (n, k) array: per column w_ours * rank(ours) + (1 - w_ours) * rank(raptor), each
        ranked over its own non-failed rows; a row one side failed takes the other side's
        rank, a row both failed is 0.5.
    """
    out = np.zeros(ours.shape, dtype=np.float64)
    for j in range(ours.shape[1]):
        ro = s13_rank_on(ours[:, j], ~ours_fail)
        rr = s13_rank_on(raptor[:, j], ~raptor_fail)
        b = w_ours * ro + (1.0 - w_ours) * rr
        b[raptor_fail] = ro[raptor_fail]
        b[ours_fail] = rr[ours_fail]
        b[ours_fail & raptor_fail] = 0.5
        out[:, j] = b
    return out


_s7_t = time.time()
_labels = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_sub_out = S7["work_dir"] / "submission.csv"
_sample = pd.read_csv(_s7_comp_root() / "sample_submission.csv", dtype={"StudyInstanceUID": str})
if S7["dry_run"] and S7["dry_run_studies"]:
    _sample = _sample.iloc[:S7["dry_run_studies"]].reset_index(drop=True)
_ids = _sample["StudyInstanceUID"].astype(str).tolist()
_cols = [c for c in _sample.columns if c != "StudyInstanceUID"]
assert sorted(_cols) == sorted(_labels), f"sample_submission columns differ: {_cols}"
if not S7["ours_ok"]:
    raise RuntimeError(f"our stage produced no table: {S7['ours_error']}")

# Our side: the family blend, and which studies at least one member scored.
_ours = pd.read_csv(S7["ours_csv"], dtype={"StudyInstanceUID": str})
_member_preds = np.load(S7["work_dir"] / "_s7_member_preds.npy")
assert _member_preds.shape[1] == len(_ours), (_member_preds.shape, len(_ours))
_scored = pd.Series(np.isfinite(_member_preds).all(axis=2).any(axis=0), index=_ours["StudyInstanceUID"])
_ours_tab = _ours.set_index("StudyInstanceUID").reindex(_ids)[_cols].to_numpy(np.float64)
_ours_fail = ~_scored.reindex(_ids, fill_value=False).to_numpy(bool)
assert np.isfinite(_ours_tab[~_ours_fail]).all(), "non-finite score on a study ours scored"

# Raptor's side, validated; anything wrong with it sends the submission to our stack alone.
_rap_tab = _rap_fail = None
if S13["raptor_ok"]:
    try:
        _raw = pd.read_csv(S13["work"] / "raptor_raw.csv", dtype={"StudyInstanceUID": str})
        _rsub = pd.read_csv(S13["work"] / "raptor_sub.csv", dtype={"StudyInstanceUID": str})
        for _df, _nm in ((_raw, "raptor_raw"), (_rsub, "raptor_sub")):
            if _df["StudyInstanceUID"].duplicated().any():
                raise ValueError(f"{_nm}: duplicate StudyInstanceUID")
            if not set(_ids) <= set(_df["StudyInstanceUID"]):
                raise ValueError(f"{_nm}: {len(set(_ids) - set(_df['StudyInstanceUID']))} studies missing")
            if not set(_cols) <= set(_df.columns):
                raise ValueError(f"{_nm}: columns differ: {list(_df.columns)}")
        _rap_tab = _raw.set_index("StudyInstanceUID").reindex(_ids)[_cols].to_numpy(np.float64)
        _rsub_tab = _rsub.set_index("StudyInstanceUID").reindex(_ids)[_cols].to_numpy(np.float64)
        if not (np.isfinite(_rap_tab).all() and ((_rap_tab >= 0) & (_rap_tab <= 1)).all()):
            raise ValueError("raptor_raw: a value is non-finite or outside [0, 1]")
        _rap_fail = (_rap_tab == 0.5).all(axis=1)       # Raptor leaves a failed study at 0.5 everywhere
        if _rap_fail.all():
            raise ValueError("Raptor failed on every study")
        if len(_ids) >= 3:
            _rho = [pd.Series(_rap_tab[:, j]).corr(pd.Series(_rsub_tab[:, j]), method="spearman")
                    for j in range(len(_cols))]
            print(f"Raptor raw vs its own rank table: min Spearman {np.nanmin(_rho):.4f}")
            if np.nanmin(_rho) < 0.99:
                raise ValueError(f"raptor_raw does not match raptor_sub: {np.round(_rho, 4).tolist()}")
    except Exception as exc:
        S13["raptor_ok"] = False
        S13["raptor_error"] = f"table: {type(exc).__name__}: {exc}"
        _rap_tab = _rap_fail = None

if _rap_tab is not None:
    _final = s13_blend(_ours_tab, _ours_fail, _rap_tab, _rap_fail, S13["w_ours"])
    S7["mode"] = (f"blend: {S13['w_ours']:.2f} ours ({S7.get('n_members_run')} members) + "
                  f"{1 - S13['w_ours']:.2f} Raptor")
    print(f"Raptor failed on {int(_rap_fail.sum())} | ours failed on {int(_ours_fail.sum())} | "
          f"both on {int((_rap_fail & _ours_fail).sum())} of {len(_ids)} studies")
    if _rap_fail.any() or _ours_fail.any():
        print(f"BLEND_PARTIAL ours_failed={int(_ours_fail.sum())} raptor_failed={int(_rap_fail.sum())} "
              f"both_failed={int((_rap_fail & _ours_fail).sum())} of {len(_ids)}")
    _both = ~_rap_fail & ~_ours_fail
    if _both.sum() >= 3:
        _rho = [pd.Series(_ours_tab[_both, j]).corr(pd.Series(_rap_tab[_both, j]), method="spearman")
                for j in range(len(_cols))]
        print("Spearman ours vs Raptor per finding: "
              + ", ".join(f"{c} {r:.3f}" for c, r in zip(_cols, _rho)) + f" | mean {np.nanmean(_rho):.3f}")
else:
    _final = s7_rank(_ours, _ids, _cols)                 # submission 11b's own write
    S7["mode"] = f"BLEND_FALLBACK raptor | ours: {S7.get('n_members_run')} members"
    print("=" * 78)
    print(f"BLEND_FALLBACK raptor: the submission is our stack alone | {S13['raptor_error']}")
    print("=" * 78)

out = _sample[["StudyInstanceUID"]].copy()
for _j, _c in enumerate(_cols):
    out[_c] = _final[:, _j]
out = out[["StudyInstanceUID", *_cols]]
out.to_csv(_sub_out, index=False)
S7["times"]["blend"] = time.time() - _s7_t
print(f"{S7['mode']} | {out.shape} -> {_sub_out}")

In [ ]:
# Sanity checks on the file Kaggle will score. A constant column scores 0.5 AUC.
chk = pd.read_csv(_sub_out, dtype={"StudyInstanceUID": str})
_test_ids = pd.read_csv(_s7_comp_root() / "test.csv", dtype={"StudyInstanceUID": str})["StudyInstanceUID"].tolist()
if S7["dry_run"] and S7["dry_run_studies"]:
    _test_ids = _test_ids[:S7["dry_run_studies"]]
nunique = chk[_cols].nunique()
print("distinct values per label:\n", nunique.to_string())
assert chk.columns.tolist() == _sample.columns.tolist(), "column order differs from sample_submission"
assert len(_cols) == 12 and chk.shape[1] == 13, f"{chk.shape[1]} columns"
assert len(chk) == len(_sample), f"{len(chk)} rows vs {len(_sample)} in sample_submission"
assert chk["StudyInstanceUID"].tolist() == _ids, "row order differs from sample_submission"
assert len(chk) == len(_test_ids) and set(chk["StudyInstanceUID"]) == set(_test_ids), "ids differ from test.csv"
assert not chk["StudyInstanceUID"].duplicated().any(), "duplicate StudyInstanceUID"
assert not chk[_cols].isna().any().any(), "NaN in submission"
assert ((chk[_cols] >= 0) & (chk[_cols] <= 1)).all().all(), "score outside [0, 1]"
if len(chk) > 5:
    assert (nunique > 1).all(), f"constant column(s): {nunique[nunique <= 1].index.tolist()}"
if S7["dropped"]:
    print("\nDROPPED TO FIT OR AFTER A FAILURE:")
    for _line in S7["dropped"]:
        print("  " + _line)
print(f"\nOK | {len(chk)} rows | 12 labels | ids match test.csv | {S7['mode']}")

In [ ]:
# Wall time per stage, and a projection to the hidden test (~1,300 studies).
N_HIDDEN = 1300
_total = time.time() - S13["t0"]
print(f" raptor: {S13['times']['raptor'] / 60:7.1f} min | ok={S13['raptor_ok']} | error={S13['raptor_error']}")
for _k in ("plan", "cache", "ours", "blend"):
    if _k in S7["times"]:
        print(f"{_k:>7}: {S7['times'][_k] / 60:7.1f} min")
print(f"  total: {_total / 60:7.1f} min = {_total / 3600:.2f} h of the {BUDGET_H} h deadline")
if "build" in S7 and S7["n_studies"].get("cache"):
    print(f"cache build: {S7['build']['wall'] / S7['n_studies']['cache']:.2f} s/study wall -> "
          f"{S7['build']['wall'] / S7['n_studies']['cache'] * N_HIDDEN / 3600:.2f} h for {N_HIDDEN} "
          f"(small runs overstate it)")
if "run" in S7:
    print(f"scoring: {S7['run']['members']} members x {len(S7['run']['offsets'])} TTA, "
          f"projected {S7['run']['projected_h']:.2f} h for the remaining studies of this run")
print(f"\nkernel done in {_total / 3600:.2f} h | {S7.get('mode')}")